In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION                        ║
# ║         ← Edit these values, then run all cells →       ║
# ╚══════════════════════════════════════════════════════════╝

# Point to the log folder here.
FOLDER = "/Users/aleblebjian/Desktop/86b4e766f78a31cbb6ca2b4f364e5a7160088afa4841d2fd9172841d9d61cfcb"

# ── CPU / Memory / Overwrite parameters ───────────────────────
THRESHOLD             = 75    # flag CPU readings strictly above this %
MEMORY_THRESHOLD      = 85    # flag Memory readings strictly above this % (native_utils "Memory %" row)
CONSECUTIVE           = 5     # min back-to-back readings to count as a spike (used for both CPU and Memory)
OVERWRITE_THRESHOLD   = 5     # flag if Overwrite Cnt > Write Cnt × (this / 100)


# ── Time window ─────────────────────────────────────────────
# Leave as None to search the whole file.
# Format: "MM/DD/YY HH:MM:SS"
# START_TIME = "06/26/26 00:00:01"
START_TIME = None
END_TIME   = None


# ── Report sort order ─────────────────────────────────────────
# True  = newest first (descending) — the default.
# False = oldest first (ascending).
# Applies to every table/timeline in the report on initial load; each pane
# still has its own sort button to flip it interactively afterward.
SORT_DESCENDING = True


# ── Last Calls tab ────────────────────────────────────────────
# Number of most-recent calls (per device type) to build a detailed,
# time-windowed mini-timeline for in the "Last Calls" tab.
LAST_N_CALLS = 5


# ── Version check ───────────────────────────────────────────
# The latest Krisp app version available — update this manually as new
# versions ship. The Summary tab compares it against the version the user
# was actually running (detected from the last "Start new session" log line)
# and flags the user's version in red if it doesn't match.
LATEST_KRISP_VERSION = "2.80.12"


In [ ]:
# ── Driver check + helper functions – no need to edit below this line ──────
import re
import json as _json_mod
import struct as _struct
from pathlib import Path
from datetime import datetime, timedelta


def get_wav_sample_rate(path):
    """
    Read the sample rate straight out of a WAV file's 'fmt ' chunk.
    Parsed manually (instead of the stdlib `wave` module) because `wave`
    raises "unknown format" on anything that isn't plain integer PCM —
    Krisp's before/after_proc.wav captures are IEEE-float PCM, which `wave`
    can't open at all. Returns an int (Hz), or None if the file is missing
    or doesn't look like a valid WAV.
    """
    try:
        with open(path, "rb") as f:
            header = f.read(256)
        riff, _size, wave_id = _struct.unpack("<4sI4s", header[0:12])
        if riff != b"RIFF" or wave_id != b"WAVE":
            return None
        pos = 12
        while pos < len(header) - 8:
            chunk_id, chunk_size = _struct.unpack("<4sI", header[pos:pos + 8])
            if chunk_id == b"fmt ":
                fmt_data = header[pos + 8: pos + 8 + chunk_size]
                if len(fmt_data) < 16:
                    return None
                _fmt_tag, _channels, sample_rate = _struct.unpack("<HHI", fmt_data[:8])
                return sample_rate
            pos += 8 + chunk_size + (chunk_size % 2)
    except (OSError, _struct.error):
        return None
    return None

TIMESTAMP_FORMAT = "%m/%d/%y %H:%M:%S"

# ── Patterns: native_utils (CPU) ────────────────────────────
RE_DUMP_HEADER = re.compile(
    r"\[(\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2})\.(\d+)\].*HardwareUtilizationMonitor::DumpUtilizationData"
)
RE_CPU_ROW = re.compile(r"\|\s*CPU\s*%\s*\|(.*)\|")
RE_MEM_ROW = re.compile(r"\|\s*Memory\s*%\s*\|(.*)\|")

# ── Patterns: kr_media_sp (Overwrite) ───────────────────────
RE_CAPTURER_STOP = re.compile(
    r"\[(\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2})\.(\d+)\].*Capturer::Stop\(\)"
)
RE_WRITE     = re.compile(r"Write\s*-\s*Cnt:\s*(\d+)")
RE_OVERWRITE = re.compile(r"Overwrite\s*-\s*Cnt:\s*(\d+)")

# ── Patterns: kr_audio_dm (Volume) ──────────────────────────
RE_VOLUME_CHANGED = re.compile(
    r"\[(\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2})\.(\d+)\]"
    r".*\[VOLUME_CHANGED\].*"
    r"Volume:\s*([\d.]+).*"
    r"OtherDeviceVolume:\s*([\d.]+).*"
    r",\s*Type:\s*(\d+)"
)


def parse_ts(s):
    return datetime.strptime(s, TIMESTAMP_FORMAT)


TIMESTAMP_FULL_FORMAT = TIMESTAMP_FORMAT + ".%f"

def parse_ts_us(sec_str, frac_str=None):
    """
    Parse a timestamp with microsecond precision for internal sorting/window
    comparisons. sec_str is the "MM/DD/YY HH:MM:SS" part (same string used
    for display); frac_str is the fractional-seconds digits from the log
    line (e.g. "912552"). Falls back to second precision if frac_str is
    missing/unparseable — never raises.
    Display strings (ts_str) are untouched everywhere; only the internal
    ts_dt gains microsecond precision so same-second events sort and
    window-filter correctly.
    """
    if frac_str:
        try:
            frac = (frac_str + "000000")[:6]   # pad/truncate to exactly 6 digits
            return datetime.strptime(sec_str + "." + frac, TIMESTAMP_FULL_FORMAT)
        except ValueError:
            pass
    return parse_ts(sec_str)


# ── CPU functions ────────────────────────────────────────────
def extract_cpu_blocks(file_path):
    """Yield (ts_str, ts_dt, [cpu_values]) for every DumpUtilizationData block."""
    current_ts_str = current_ts_dt = None
    with open(file_path, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            m = RE_DUMP_HEADER.search(line)
            if m:
                current_ts_str = m.group(1)
                current_ts_dt  = parse_ts_us(current_ts_str, m.group(2))
                continue
            if current_ts_str:
                m2 = RE_CPU_ROW.search(line)
                if m2:
                    values = [float(v.strip()) for v in m2.group(1).split("|") if v.strip()]
                    yield current_ts_str, current_ts_dt, values
                    current_ts_str = current_ts_dt = None


def collect_cpu_readings(files, start_dt, end_dt):
    readings = []
    total_blocks = in_range_blocks = 0
    for fp in files:
        for ts_str, ts_dt, values in extract_cpu_blocks(fp):
            total_blocks += 1
            if start_dt and ts_dt < start_dt:
                continue
            if end_dt and ts_dt > end_dt:
                continue
            in_range_blocks += 1
            for i, v in enumerate(values):
                readings.append((ts_str, ts_dt, i, v))
    return readings, total_blocks, in_range_blocks


def find_cpu_spikes(readings, threshold, consecutive):
    spikes, run = [], []
    for entry in readings:
        if entry[3] > threshold:
            run.append(entry)
        else:
            if len(run) >= consecutive:
                spikes.append(run[:])
            run = []
    if len(run) >= consecutive:
        spikes.append(run[:])
    return spikes


# ── Memory functions (native_utils "Memory %" row, same block as CPU) ──
def extract_mem_blocks(file_path):
    """Yield (ts_str, ts_dt, [mem_values]) for every DumpUtilizationData block."""
    current_ts_str = current_ts_dt = None
    with open(file_path, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            m = RE_DUMP_HEADER.search(line)
            if m:
                current_ts_str = m.group(1)
                current_ts_dt  = parse_ts_us(current_ts_str, m.group(2))
                continue
            if current_ts_str:
                m2 = RE_MEM_ROW.search(line)
                if m2:
                    values = [float(v.strip()) for v in m2.group(1).split("|") if v.strip()]
                    yield current_ts_str, current_ts_dt, values
                    current_ts_str = current_ts_dt = None


def collect_memory_readings(files, start_dt, end_dt):
    readings = []
    total_blocks = in_range_blocks = 0
    for fp in files:
        for ts_str, ts_dt, values in extract_mem_blocks(fp):
            total_blocks += 1
            if start_dt and ts_dt < start_dt:
                continue
            if end_dt and ts_dt > end_dt:
                continue
            in_range_blocks += 1
            for i, v in enumerate(values):
                readings.append((ts_str, ts_dt, i, v))
    return readings, total_blocks, in_range_blocks


# find_mem_spikes reuses find_cpu_spikes — same threshold/consecutive-run logic.


# ── Overwrite functions ──────────────────────────────────────
def extract_overwrite_blocks(file_path):
    """
    Yield (ts_str, ts_dt, write_cnt, overwrite_cnt, write_line, overwrite_line)
    for every Capturer::Stop block that contains Write + Overwrite rows.
    """
    current_ts_str = current_ts_dt = None
    write_cnt = overwrite_cnt = None
    write_line = overwrite_line = None

    with open(file_path, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            m = RE_CAPTURER_STOP.search(line)
            if m:
                current_ts_str = m.group(1)
                current_ts_dt  = parse_ts_us(current_ts_str, m.group(2))
                write_cnt = overwrite_cnt = None
                write_line = overwrite_line = None
                continue

            if current_ts_str:
                mw = RE_WRITE.search(line)
                if mw and "Overwrite" not in line:
                    write_cnt  = int(mw.group(1))
                    write_line = line.rstrip()

                mo = RE_OVERWRITE.search(line)
                if mo:
                    overwrite_cnt  = int(mo.group(1))
                    overwrite_line = line.rstrip()

                if write_cnt is not None and overwrite_cnt is not None:
                    yield (current_ts_str, current_ts_dt,
                           write_cnt, overwrite_cnt,
                           write_line, overwrite_line)
                    current_ts_str = current_ts_dt = None
                    write_cnt = overwrite_cnt = None


def find_overwrite_flags(files, overwrite_pct, start_dt, end_dt):
    flagged = []
    total_blocks = in_range_blocks = 0
    for fp in files:
        for ts_str, ts_dt, w_cnt, ow_cnt, w_line, ow_line in extract_overwrite_blocks(fp):
            total_blocks += 1
            if start_dt and ts_dt < start_dt:
                continue
            if end_dt and ts_dt > end_dt:
                continue
            in_range_blocks += 1
            if w_cnt > 0 and ow_cnt > (w_cnt * overwrite_pct / 100):
                ratio = (ow_cnt / w_cnt) * 100
                flagged.append((ts_str, ts_dt, w_cnt, ow_cnt, ratio, w_line, ow_line, fp))
    return flagged, total_blocks, in_range_blocks


# ── Volume functions ─────────────────────────────────────────
def collect_volume_events(files, vol_threshold, start_dt, end_dt, device_types=None):
    """
    Scan kr_audio_dm files for VOLUME_CHANGED lines containing ", Type: 0" or ", Type: 1".
    Flag each event where Volume * OtherDeviceVolume < vol_threshold.
    device_types: set of int (0=Mic, 1=Speaker). None means collect all.
    Returns (flagged_events, total_events, in_range_events) where each event is a dict.
    """
    flagged = []
    total_events = in_range_events = 0

    for fp in files:
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            for line in fh:
                if ", Type: " not in line or "[VOLUME_CHANGED]" not in line:
                    continue
                m = RE_VOLUME_CHANGED.search(line)
                if not m:
                    continue
                ts_str    = m.group(1)
                ts_dt     = parse_ts_us(ts_str, m.group(2))
                volume    = float(m.group(3))
                other_vol = float(m.group(4))
                dev_type  = int(m.group(5))   # 0 = Microphone, 1 = Speaker

                if device_types is not None and dev_type not in device_types:
                    continue
                total_events += 1
                if start_dt and ts_dt < start_dt:
                    continue
                if end_dt and ts_dt > end_dt:
                    continue
                in_range_events += 1

                product = volume * other_vol
                if product < vol_threshold:
                    flagged.append({
                        "ts_str":    ts_str,
                        "ts_dt":     ts_dt,
                        "volume":    volume,
                        "other_vol": other_vol,
                        "product":   product,
                        "dev_type":  dev_type,
                        "raw_line":  line.rstrip(),
                        "file":      fp,
                    })

    return flagged, total_events, in_range_events


# ── File resolution ──────────────────────────────────────────
def _rotation_sort_key(p):
    """
    Sort rotated log files oldest-first:
      kr_audio_dm.10.log → kr_audio_dm.9.log → ... → kr_audio_dm.1.log → kr_audio_dm.log
    Files with a numeric suffix come first (descending by number = oldest first).
    The bare file (no numeric suffix) sorts last (newest).
    """
    stem = p.stem          # e.g. "kr_audio_dm.10" or "kr_audio_dm"
    parts = stem.rsplit(".", 1)
    if len(parts) == 2 and parts[1].isdigit():
        return (0, -int(parts[1]))   # numbered: sort descending by number
    return (1, 0)                    # bare name: always last


def resolve_files(folder, pattern):
    p = Path(folder)
    if not p.is_dir():
        raise ValueError(f"FOLDER '{folder}' is not a valid directory.")
    logs_dir = p / "Logs"
    search_in = logs_dir if logs_dir.is_dir() else p
    found = sorted(search_in.glob(pattern), key=_rotation_sort_key)
    return found


# ── Driver / dm_reporting check ──────────────────────────────
def parse_device_map(folder):
    """
    Build { device_id: device_name } from dm_reporting.txt first,
    then fill any gaps from dumpDeviceList tables in kr_app logs.
    """
    import re as _re
    device_map = {}
    p = Path(folder)

    # 1. dm_reporting.txt
    candidates = list(p.glob("dm_reporting.txt")) + list(p.glob("Logs/dm_reporting.txt"))
    if candidates:
        _re_dev = _re.compile(r"idx:\s*(\{[^}]+\}\.\{[^}]+\}).*?name:\s*'([^']+)'")
        with open(candidates[0], "r", encoding="utf-8", errors="replace") as fh:
            for line in fh:
                m = _re_dev.search(line)
                if m:
                    device_map[m.group(1).strip()] = m.group(2).strip()

    # 2. kr_app dumpDeviceList tables — fills any IDs missing from dm_reporting
    logs_dir = p / "Logs"
    search_in = logs_dir if logs_dir.is_dir() else p
    kr_app_files = sorted(search_in.glob("*kr_app*.log")) + sorted(search_in.glob("*kr_app*.txt"))
    _re_id_col = _re.compile(r"^(\{[^}]+\}\.\{[^}]+\})\s*\|")
    for fp in kr_app_files:
        in_table = False
        header_seen = False
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            for line in fh:
                line_s = line.rstrip()
                if "DeviceManager.dumpDeviceList" in line_s:
                    in_table = True
                    header_seen = False
                    continue
                if not in_table:
                    continue
                # separator line
                if line_s.startswith("---"):
                    continue
                # header row (contains "id" and "name" as column headers)
                if not header_seen and "| name" in line_s and line_s.strip().startswith("id"):
                    header_seen = True
                    continue
                # data row: starts with a device ID
                m = _re_id_col.match(line_s.strip())
                if m:
                    cols = [c.strip() for c in line_s.split("|")]
                    if len(cols) >= 3:
                        dev_id   = cols[0].strip()
                        dev_name = cols[2].strip()  # name is 3rd column (index 2)
                        if dev_id and dev_name and dev_id not in device_map:
                            device_map[dev_id] = dev_name
                elif header_seen and line_s.startswith("["):
                    # hit a new log line — end of table
                    in_table = False
                    header_seen = False

    return device_map


def collect_device_report(folder):
    """
    Build two views of audio devices, per type, for the Headsets tab — keyed
    by device NAME (not device ID). The same physical headset can re-enumerate
    under a new device ID each time it reconnects, so grouping by ID would
    show the same headset multiple times; grouping by name collapses those.
      submitted_mic/spk : set of device names actually connected at the
                           moment the report was generated (dm_reporting.txt's
                           live DeviceContainer snapshot — dm_reporting.txt is
                           written fresh right before the customer submits,
                           so this is "what was plugged in at submission time").
      other_mic/spk      : set of device names seen anywhere else across the
                           session (kr_app dumpDeviceList tables, which are
                           periodic full device-enumeration dumps) that are
                           NOT among the submission-time names — i.e. things
                           the customer had connected earlier but
                           unplugged/switched away from before submitting.
    Returns (submitted_mic, submitted_spk, other_mic, other_spk), each a
    sorted list of names.
    """
    p = Path(folder)
    submitted_mic, submitted_spk = set(), set()

    # ── 1. dm_reporting.txt DeviceContainer snapshot (submission-time truth) ──
    candidates = list(p.glob("dm_reporting.txt")) + list(p.glob("Logs/dm_reporting.txt"))
    if candidates:
        text = candidates[0].read_text(encoding="utf-8", errors="replace")
        _re_dev = re.compile(r"name:\s*'([^']+)'")
        in_container = False
        current_type = None
        for raw_line in text.splitlines():
            s = raw_line.strip()
            if s == "--- DeviceContainer ---":
                in_container = True
                current_type = None
                continue
            if s.startswith("---"):
                in_container = False
                current_type = None
                continue
            if not in_container:
                continue
            if s.startswith("DeviceType:"):
                t = s.split(":", 1)[1].strip().lower()
                current_type = 0 if t == "microphone" else (1 if t == "speaker" else None)
                continue
            m = _re_dev.search(s)
            if m and current_type is not None:
                dev_name = m.group(1).strip()
                (submitted_mic if current_type == 0 else submitted_spk).add(dev_name)

    # ── 2. kr_app dumpDeviceList tables — everything ever seen, by type ──
    seen_mic, seen_spk = set(submitted_mic), set(submitted_spk)
    logs_dir = p / "Logs"
    search_in = logs_dir if logs_dir.is_dir() else p
    kr_app_files = sorted(search_in.glob("*kr_app*.log")) + sorted(search_in.glob("*kr_app*.txt"))
    _re_header = re.compile(r"DeviceManager\.dumpDeviceList\.\s*Type:\s*(\d+)")
    _re_id_col = re.compile(r"^(\{[^}]+\}\.\{[^}]+\})\s*\|")
    for fp in kr_app_files:
        in_table = False
        header_seen = False
        table_type = None
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            for line in fh:
                line_s = line.rstrip()
                hm = _re_header.search(line_s)
                if hm:
                    in_table = True
                    header_seen = False
                    table_type = int(hm.group(1))
                    continue
                if not in_table:
                    continue
                if line_s.startswith("---"):
                    continue
                if not header_seen and "| name" in line_s and line_s.strip().startswith("id"):
                    header_seen = True
                    continue
                m = _re_id_col.match(line_s.strip())
                if m:
                    cols = [c.strip() for c in line_s.split("|")]
                    if len(cols) >= 3:
                        dev_name = cols[2].strip()
                        if dev_name and table_type in (0, 1):
                            (seen_mic if table_type == 0 else seen_spk).add(dev_name)
                elif header_seen and line_s.startswith("["):
                    in_table = False
                    header_seen = False
                    table_type = None

    other_mic = sorted(seen_mic - submitted_mic, key=str.lower)
    other_spk = sorted(seen_spk - submitted_spk, key=str.lower)

    return sorted(submitted_mic, key=str.lower), sorted(submitted_spk, key=str.lower), other_mic, other_spk


def check_dm_reporting(folder):
    """
    Return (is_empty, report_path).
    is_empty = True  if the file is missing or completely empty (no content at all).
    is_empty = False if the file exists and has any content — analysis proceeds normally.
    """
    p = Path(folder)
    candidates = (
        list(p.glob("dm_reporting.txt")) +
        list(p.glob("Logs/dm_reporting.txt")) +
        list((p / "Logs").glob("dm_reporting.txt") if (p / "Logs").is_dir() else [])
    )
    if not candidates:
        return True, None
    rpt = candidates[0]
    content = rpt.read_text(encoding="utf-8", errors="replace").strip()
    return (len(content) == 0), rpt


# ── Photos/videos attached to the report ───────────────────────
# Customers occasionally drag screenshots or screen recordings into the same
# folder they zip up and submit. These sit at the TOP LEVEL of the report
# folder (next to systeminfo.json, description.txt, etc.) — not inside Logs.
_IMAGE_EXTS = {
    ".jpg", ".jpeg", ".png", ".gif", ".bmp", ".webp", ".heic", ".heif",
    ".tiff", ".tif", ".svg",
}
_VIDEO_EXTS = {
    ".mp4", ".mov", ".avi", ".mkv", ".wmv", ".webm", ".m4v", ".3gp", ".flv",
}


def find_top_level_media_files(folder):
    """
    Scan only the top-level files of `folder` (not recursively, and not the
    Logs subfolder) for image/video extensions. Returns (images, videos),
    each a list of filenames, sorted.
    """
    p = Path(folder)
    images, videos = [], []
    if not p.is_dir():
        return images, videos
    for entry in p.iterdir():
        if not entry.is_file():
            continue
        ext = entry.suffix.lower()
        if ext in _IMAGE_EXTS:
            images.append(entry.name)
        elif ext in _VIDEO_EXTS:
            videos.append(entry.name)
    return sorted(images), sorted(videos)


# ── Patterns: kr_app (System) ────────────────────────────────
RE_KR_APP_TS        = re.compile(r"\[(\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2})\.(\d+)\]")
RE_AR_START         = re.compile(
    r"ARHistory\[microphone\]\.startModel: Model is started\. ARDescriptor:\s*(\{.*?\})"
)
RE_UNHEALTHY        = re.compile(r"unhealthy", re.IGNORECASE)
RE_OVERLOAD         = re.compile(
    r"\[(\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2})\.(\d+)\].*"
    r"\[SYSTEM_OVERLOAD\].*FilterId:\s*(\w+),\s*State:\s*(\w+)"
)
RE_MUTE_CHANGE = re.compile(
    r"\[(\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2})\.\d+\]"
)  # timestamp only; mute parsing done per-line below



def collect_kr_app_system(files, start_dt, end_dt):
    """
    Scan kr_app files and return:
      unhealthy_events : list of (ts_str, raw_line)
      overload_events  : list of (ts_str, ts_dt, filter_id, state, raw_line)
      mute_events      : list of (ts_str, ts_dt, dev_type, device_name, from_muted, to_muted, raw_line)
    """
    import json as _j, re as _re

    unhealthy_events = []
    overload_events  = []
    mute_events      = []

    _re_name_from_to = _re.compile(
        r'"name":"([^"]+)"[^}]*"from":\{([^}]*)\},"to":\{([^}]*)\}'
    )

    _last_device_name = {}   # {device_type_int: last_seen_name}
    for fp in files:
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            for line in fh:
                line_s = line.rstrip()

                ts_m = RE_KR_APP_TS.search(line_s)
                if not ts_m:
                    continue
                try:
                    ts_dt  = parse_ts_us(ts_m.group(1), ts_m.group(2))
                    ts_str = ts_m.group(1)
                except ValueError:
                    continue

                if start_dt and ts_dt < start_dt:
                    continue
                if end_dt   and ts_dt > end_dt:
                    continue

                # track last seen device name per type from getGraphSettings
                if "AudioService:getGraphSettings:" in line_s:
                    import re as _rgs, json as _jgs
                    _dt_m = _rgs.search(r"deviceType:\s*(\d+)", line_s)
                    _di_m = _rgs.search(r"deviceInfo:\s*(\{.*\})", line_s)
                    if _dt_m and _di_m:
                        try:
                            _di = _jgs.loads(_di_m.group(1))
                            _last_device_name[int(_dt_m.group(1))] = _di.get("name", "")
                        except Exception:
                            pass

                # 1. unhealthy — working device only
                if "working device is unhealthy" in line_s:
                    # determine device type from streamType
                    _st_m = _re2.search(r"streamType:\s*(\S+)", line_s)
                    _dev_name = ""
                    if _st_m:
                        _st = _st_m.group(1).lower()
                        _dtype = 0 if "mic" in _st else (1 if "spk" in _st else None)
                        if _dtype is not None:
                            _dev_name = _last_device_name.get(_dtype, "")
                    unhealthy_events.append((ts_str, line_s, _dev_name))

                # 2. system overload
                m = RE_OVERLOAD.search(line_s)
                if m:
                    overload_events.append((m.group(1), parse_ts_us(m.group(1), m.group(2)),
                                            m.group(3), m.group(4), line_s))

                # 3. mute state change via DEVICE_LIST_CHANGE Modified[]
                if "[DEVICE_LIST_CHANGE]" not in line_s or "isMuted" not in line_s:
                    continue

                # extract Type
                type_m = _re.search(r"Type:\s*(microphone|speaker)", line_s)
                if not type_m:
                    continue
                dev_type = type_m.group(1)

                # find all Modified entries that carry an isMuted transition
                for m2 in _re_name_from_to.finditer(line_s):
                    device_name = m2.group(1)
                    from_str    = m2.group(2)
                    to_str      = m2.group(3)
                    if "isMuted" not in from_str or "isMuted" not in to_str:
                        continue
                    from_muted = '"isMuted":true'  in from_str or "'isMuted':true"  in from_str
                    to_muted   = '"isMuted":true'  in to_str   or "'isMuted':true"  in to_str
                    if from_muted == to_muted:
                        continue
                    mute_events.append((
                        ts_str, ts_dt, dev_type, device_name,
                        from_muted, to_muted, line_s
                    ))

    # 4. ducking — collect all (prev_line, ducking_line) pairs; caller takes the last one
    ducking_events = []
    for fp in files:
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            prev_line = ""
            for line in fh:
                line_s = line.rstrip()
                if "AudioService:onAudioStreamCreated: Ducking mode:" in line_s:
                    ts_m = RE_KR_APP_TS.search(line_s)
                    if not ts_m:
                        prev_line = line_s
                        continue
                    try:
                        ts_dt  = parse_ts_us(ts_m.group(1), ts_m.group(2))
                        ts_str = ts_m.group(1)
                    except ValueError:
                        prev_line = line_s
                        continue
                    if start_dt and ts_dt < start_dt:
                        prev_line = line_s
                        continue
                    if end_dt and ts_dt > end_dt:
                        prev_line = line_s
                        continue
                    # check if previous line is a getDuckingMode error
                    error_line = prev_line if "getDuckingMode" in prev_line and ":[E]:" in prev_line else None
                    ducking_events.append((ts_str, ts_dt, error_line, line_s))
                prev_line = line_s

    # 5. session starts — "Start new session" + first/second instance
    session_events = []
    for fp in files:
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            lines_list = fh.readlines()
        for idx_l, line in enumerate(lines_list):
            line_s = line.rstrip()
            if "Start new session" not in line_s:
                continue
            ts_m = RE_KR_APP_TS.search(line_s)
            if not ts_m:
                continue
            try:
                ts_dt  = parse_ts_us(ts_m.group(1), ts_m.group(2))
                ts_str = ts_m.group(1)
            except ValueError:
                continue
            if start_dt and ts_dt < start_dt:
                continue
            if end_dt and ts_dt > end_dt:
                continue
            # scan the next ~20 lines for instance info
            instance = ""
            for ll in lines_list[idx_l+1:idx_l+25]:
                if "launched as a first instance" in ll:
                    instance = "First Instance"
                    break
                if "launched as a second instance" in ll:
                    instance = "Second Instance"
                    break
            session_events.append((ts_str, ts_dt, instance, line_s))

    return unhealthy_events, overload_events, mute_events, ducking_events, session_events


# ── Windows username from the last "Start new session" ────────
# Right after a new session starts, kr_app re-initializes its logger, which
# logs a line like:
#   Logger::.ctor(). Logger created. FilePath: C:\Users\MPilapil\AppData\Local\KrispEnt\Logs\kr_app.log, Level: 2
# The folder right after "Users\" is the Windows account name.
RE_LOGGER_CREATED_PATH = re.compile(
    r"Logger created\.\s*FilePath:\s*[A-Za-z]:\\Users\\([^\\]+)\\"
)
_SESSION_USERNAME_LOOKAHEAD = 40   # max lines to scan after "Start new session"


# ── Krisp app version from the last "Start new session" line ──
# The same line that announces a new session also carries the running
# version, e.g.:
#   * Start new session (version: 2.77.5) (timezone: -05:00) *
RE_SESSION_VERSION = re.compile(r"Start new session\s*\(version:\s*([^)]+)\)")


def extract_last_session_version(files):
    """
    Find the chronologically LAST "Start new session" line across all kr_app
    files and return the captured version string (e.g. "2.77.5"), or None if
    no session-start line with a version was found.
    """
    best = None   # (ts_dt, version)
    for fp in files:
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            for line in fh:
                if "Start new session" not in line:
                    continue
                ver_m = RE_SESSION_VERSION.search(line)
                if not ver_m:
                    continue
                ts_m = RE_KR_APP_TS.search(line)
                if not ts_m:
                    continue
                try:
                    ts_dt = parse_ts_us(ts_m.group(1), ts_m.group(2))
                except ValueError:
                    continue
                if best is None or ts_dt >= best[0]:
                    best = (ts_dt, ver_m.group(1).strip())

    return best[1] if best else None


def extract_last_session_username(files):
    """
    Find the chronologically LAST "Start new session" line across all kr_app
    files, then scan forward a short window for the "Logger created. FilePath:"
    line and return the captured Windows username. Returns None if no session
    start (or no matching FilePath line after it) is found.
    """
    best = None   # (ts_dt, lines, idx)
    for fp in files:
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            lines = fh.readlines()
        for idx, line in enumerate(lines):
            if "Start new session" not in line:
                continue
            ts_m = RE_KR_APP_TS.search(line)
            if not ts_m:
                continue
            try:
                ts_dt = parse_ts_us(ts_m.group(1), ts_m.group(2))
            except ValueError:
                continue
            if best is None or ts_dt >= best[0]:
                best = (ts_dt, lines, idx)

    if best is None:
        return None

    _, lines, idx = best
    for look in lines[idx + 1: idx + 1 + _SESSION_USERNAME_LOOKAHEAD]:
        m = RE_LOGGER_CREATED_PATH.search(look)
        if m:
            return m.group(1)
    return None


# ── Last log-submission time ──────────────────────────────────
# Every time the customer submits a diagnostics report, kr_app writes a
# request-stats block that looks like:
#
#   [07/06/26 09:03:30.625773]:[I]:
#   ----------------------------------------------------------------
#   Request                  | Count | Errors | First Request | Last Request
#   ----------------------------------------------------------------
#   ...
#   ----------------------------------------------------------------
#
# The bracketed timestamp a couple of lines above the "Request | Count |
# Errors | ..." header is when that particular report was submitted. This
# block repeats every time a report is submitted, so we want the LAST
# (most recent) one across all kr_app files.
RE_LOG_SUBMIT_HEADER  = re.compile(r"Request\s*\|\s*Count\s*\|\s*Errors\s*\|\s*First Request\s*\|\s*Last Request")
_LOG_SUBMIT_LOOKBACK  = 5   # max lines to scan above the header for the timestamp


def extract_last_log_submission_time(files):
    """
    Find the chronologically LAST request-stats block (the report-submission
    banner) across all kr_app files and return (ts_dt, ts_str, True) for the
    timestamp line just above it.

    If no such block exists anywhere (e.g. the customer's build never wrote
    one, or the relevant lines were trimmed), fall back to the single latest
    timestamp recorded anywhere in the kr_app files and return
    (ts_dt, ts_str, False) instead — the caller can use the third element to
    label it "Last log activity" rather than "Log submitted". Only returns
    None if the files contain no parseable timestamps at all.
    """
    best = None       # (ts_dt, ts_str) — real submission-block timestamp
    latest_any = None  # (ts_dt, ts_str) — latest timestamp seen anywhere, fallback

    for fp in files:
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            lines = fh.readlines()
        for idx, line in enumerate(lines):
            ts_m = RE_KR_APP_TS.search(line)
            if ts_m:
                try:
                    ts_dt = parse_ts_us(ts_m.group(1), ts_m.group(2))
                except ValueError:
                    ts_dt = None
                if ts_dt is not None and (latest_any is None or ts_dt >= latest_any[0]):
                    latest_any = (ts_dt, ts_m.group(1))

            if not RE_LOG_SUBMIT_HEADER.search(line):
                continue
            for look in lines[max(0, idx - _LOG_SUBMIT_LOOKBACK): idx]:
                lm = RE_KR_APP_TS.search(look)
                if not lm:
                    continue
                try:
                    l_dt = parse_ts_us(lm.group(1), lm.group(2))
                except ValueError:
                    continue
                if best is None or l_dt >= best[0]:
                    best = (l_dt, lm.group(1))

    if best is not None:
        return (best[0], best[1], True)
    if latest_any is not None:
        return (latest_any[0], latest_any[1], False)
    return None


# ── Patterns: kr_app (DumpStreams + Calls) ───────────────────
import re as _re2, json as _j2

_RE_CALL_START = _re2.compile(
    r'\[(\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2})\.\d+\].*'
    r'\[CALL_START\].*?Type:\s*(microphone|speaker),\s*'
    r'AppId:\s*[^,]+,\s*AppName:\s*([^,]+),\s*'
    r'CallId:\s*([^,]+),\s*'
    r'InitialNcState:\s*(\w+),\s*'
    r'InitialMuteState:\s*(\w+),.*?'
    r'DeviceInfo:\s*(\{.*)'
)


def collect_kr_app_dumpstreams(files, start_dt, end_dt):
    """
    Collect DeviceManager.dumpStreams blocks.
    Returns two lists of entries (mic, speaker), each entry:
      (ts_str, ts_dt, [(deviceName, appName), ...])
    Empty blocks are included with an empty list.
    """
    mic_dumps = []
    spk_dumps = []

    for fp in files:
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            lines = fh.readlines()

        i = 0
        while i < len(lines):
            line = lines[i].rstrip()
            if "DeviceManager.dumpStreams. Type:" in line:
                ts_m = RE_KR_APP_TS.search(line)
                if not ts_m:
                    i += 1
                    continue
                try:
                    ts_dt  = parse_ts_us(ts_m.group(1), ts_m.group(2))
                    ts_str = ts_m.group(1)
                except ValueError:
                    i += 1
                    continue

                if start_dt and ts_dt < start_dt:
                    i += 1
                    continue
                if end_dt and ts_dt > end_dt:
                    i += 1
                    continue

                dev_type = 0 if "Type: 0" in line else 1

                # read the table lines that follow
                entries = []
                j = i + 1
                empty_block = False
                while j < len(lines):
                    tline = lines[j].rstrip()
                    if tline.strip() == "Empty":
                        empty_block = True
                        j += 1
                        continue
                    # stop when we hit a new log line (starts with '[')
                    if tline.startswith("[") and "]:[" in tline:
                        break
                    # data row: not a separator and contains '|'
                    if "|" in tline and not tline.startswith("-") and not tline.startswith("device"):
                        cols = [c.strip() for c in tline.split("|")]
                        if len(cols) >= 5:
                            device_name = cols[0]
                            app_name    = cols[3]
                            if device_name:
                                entries.append((device_name, app_name))
                    j += 1
                i = j

                record = (ts_str, ts_dt, entries)
                if dev_type == 0:
                    mic_dumps.append(record)
                else:
                    spk_dumps.append(record)
            else:
                i += 1

    return mic_dumps, spk_dumps


def collect_kr_app_calls(files, start_dt, end_dt):
    """
    Collect CALL_START and CALL_END events. "Krisp" calls are dropped
    entirely (never a real call). System Settings/rundll32 calls are OS
    noise, not real user calls, so they're dropped from mic_calls/spk_calls
    (used everywhere calls are listed/counted, e.g. the Timeline tabs) — but
    kept in mic_calls_raw/spk_calls_raw (same record shape) so a caller that
    needs to reconstruct "last N *real* calls" — walking backward and
    skipping OS noise until N real ones are found — has the data to do it
    (see _last_calls_device_tab in the report cell).

    Mic CALL_START records carry an extra trailing field, ar_language: if an
    "ARHistory[microphone].startModel: Model is started." line appears within
    the next few lines after the call start, its ARDescriptor.source value
    (the accent language, e.g. "filipino") is captured here.
    """
    OS_NOISE_APPS = {"system settings", "rundll32"}
    AR_LOOKAHEAD = 8   # max lines to look ahead for the AR startModel line
    mic_calls = []
    spk_calls = []
    mic_calls_raw = []
    spk_calls_raw = []
    call_ends  = {}

    for fp in files:
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            lines = fh.readlines()

        for idx, line in enumerate(lines):
            line_s = line.rstrip()
            is_start = "[CALL_START]" in line_s
            is_end   = "[CALL_END]"   in line_s
            if not is_start and not is_end:
                continue

            ts_m = RE_KR_APP_TS.search(line_s)
            if not ts_m:
                continue
            try:
                ts_dt  = parse_ts_us(ts_m.group(1), ts_m.group(2))
                ts_str = ts_m.group(1)
            except ValueError:
                continue
            # full timestamp with microseconds for tie-breaking sort
            _full_ts_m = _re2.search(r"\[(\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}\.\d+)\]", line_s)
            ts_full = _full_ts_m.group(1) if _full_ts_m else ts_str

            if start_dt and ts_dt < start_dt:
                continue
            if end_dt and ts_dt > end_dt:
                continue

            def _get(pattern, default=""):
                m = _re2.search(pattern, line_s, _re2.IGNORECASE)
                return m.group(1).strip() if m else default

            call_id = _get(r"CallId:\s*([^,]+)")

            if is_end:
                duration    = _get(r"Duration:\s*(\d+)")
                final_nc    = _get(r"FinalNcState:\s*(\w+)")
                final_mute  = _get(r"FinalMuteState:\s*(\w+)")
                call_ends[call_id] = (ts_str, ts_dt, duration, final_nc, final_mute, ts_full)
                continue

            app_name = _get(r"AppName:\s*([^,]+)")
            dev_type = _get(r"Type:\s*(microphone|speaker)")
            if app_name.lower() == "krisp":
                continue

            initial_nc   = _get(r"InitialNcState:\s*(\w+)")
            initial_mute = _get(r"InitialMuteState:\s*(\w+)")

            device_name = ""
            bvc_state   = ""
            sample_rate = ""
            # Two log formats seen in the wild: an older "DeviceInfo: {...}"
            # (single object) and a newer "DeviceInfoList_v2: [{...}], ModelInfoList: [...]"
            # (list, with extra fields trailing after it). raw_decode parses
            # just the leading JSON value and ignores anything after it, so
            # both formats — and any trailing text — are handled the same way.
            di_m = _re2.search(r'DeviceInfo(?:List_v2)?:\s*([\{\[].*)', line_s)
            if di_m:
                try:
                    di, _ = _j2.JSONDecoder().raw_decode(di_m.group(1))
                    if isinstance(di, list):
                        di = di[0] if di else {}
                    device_name = di.get("info", {}).get("name", "")
                    bvc_state   = di.get("bvcCompatibilityState", "")
                    sample_rate = di.get("info", {}).get("fmt", {}).get("sampleRate", "")
                except Exception:
                    device_name = _get(r'"name":"([^"]+)"')
                    bvc_state   = _get(r'"bvcCompatibilityState":"([^"]+)"')
                    sample_rate = _get(r'"sampleRate":\s*(\d+)')

            # ── Accent Conversion: look ahead for ARHistory[microphone].startModel ──
            ar_language = ""
            if dev_type == "microphone":
                for k in range(idx + 1, min(idx + 1 + AR_LOOKAHEAD, len(lines))):
                    look = lines[k]
                    if "[CALL_START]" in look or "[CALL_END]" in look:
                        break   # next call event — stop looking
                    ar_m = RE_AR_START.search(look)
                    if ar_m:
                        try:
                            ar_desc = _j2.loads(ar_m.group(1))
                            ar_language = ar_desc.get("source", "")
                        except Exception:
                            pass
                        break

            record = (ts_str, ts_dt, app_name, call_id,
                      initial_nc, initial_mute, device_name, bvc_state, ts_full, ar_language,
                      sample_rate)

            _is_os_noise = app_name.lower() in OS_NOISE_APPS
            if dev_type == "microphone":
                mic_calls_raw.append(record)
                if not _is_os_noise:
                    mic_calls.append(record)
            else:
                spk_calls_raw.append(record)
                if not _is_os_noise:
                    spk_calls.append(record)

    return mic_calls, spk_calls, call_ends, mic_calls_raw, spk_calls_raw


# ── Patterns: TestNC (last recording session — model + mic used) ────
RE_TESTNC_START = re.compile(r"TestNcService:onStartRecording: recording started successfully")
RE_CHANGE_PROC_SETTINGS = re.compile(
    r"StreamManager:changeProcessorSettings:.*ProcId:\s*kr_mic_ar,\s*Settings:\s*(\{.*\})"
)
RE_CREATE_GRAPH_INSTANCE = re.compile(
    r"StreamManager:createGraphInstance:.*source:\s*(\{.*?\}),\s*sink:\s*\{"
)
RE_GET_GRAPH_SETTINGS = re.compile(
    r"AudioService:getGraphSettings:\s*deviceType:\s*(\d+),\s*deviceInfo:\s*(\{.*)"
)
_LOOKBACK = 60   # max lines to search backward for each anchor


def collect_testnc_last(files):
    """
    Find the chronologically LAST "TestNcService:onStartRecording" line across
    all kr_app files (files must be sorted oldest → newest, as resolve_files does).
    Then scan backward from it for:
      - the nearest StreamManager:changeProcessorSettings (ProcId: kr_mic_ar) line
        → accent-conversion model name
      - the nearest StreamManager:createGraphInstance line before that
        → microphone device name (source.name)
    Returns a dict, or None if no TestNC session was found at all.
    """
    best = None   # (ts_dt, ts_str, lines, idx, fp)

    for fp in files:
        with open(fp, "r", encoding="utf-8", errors="replace") as fh:
            lines = fh.readlines()
        for idx, line in enumerate(lines):
            if not RE_TESTNC_START.search(line):
                continue
            ts_m = RE_KR_APP_TS.search(line)
            if not ts_m:
                continue
            try:
                ts_dt = parse_ts_us(ts_m.group(1), ts_m.group(2))
            except ValueError:
                continue
            if best is None or ts_dt >= best[0]:
                best = (ts_dt, ts_m.group(1), lines, idx, fp)

    if best is None:
        return None

    ts_dt, ts_str, lines, idx, fp = best

    # ── nearest changeProcessorSettings (kr_mic_ar) line above ──
    model_name  = ""
    ar_line     = ""
    ar_idx      = None
    for j in range(idx - 1, max(idx - 1 - _LOOKBACK, -1), -1):
        m = RE_CHANGE_PROC_SETTINGS.search(lines[j])
        if m:
            ar_idx  = j
            ar_line = lines[j].rstrip()
            try:
                settings = _json_mod.loads(m.group(1))
                for p in settings.get("parameters", []):
                    if p.get("name") == "modelName":
                        model_name = p.get("value", "")
                        break
            except Exception:
                pass
            break

    # ── nearest createGraphInstance line above that (mic device name) ──
    mic_name = ""
    mic_sample_rate = ""
    graph_line = ""
    search_from = ar_idx if ar_idx is not None else idx
    for j in range(search_from - 1, max(search_from - 1 - _LOOKBACK, -1), -1):
        m = RE_CREATE_GRAPH_INSTANCE.search(lines[j])
        if m:
            graph_line = lines[j].rstrip()
            try:
                src_dev = _json_mod.loads(m.group(1))
                name = src_dev.get("name", "")
                # best-effort: only present if this source object nests the
                # same "fmt" block CALL_START/CALL_END device info carries.
                sr = (src_dev.get("fmt", {}) or {}).get("sampleRate", "")
            except Exception:
                name = ""
                sr = ""
            if name:
                mic_name = name
                mic_sample_rate = sr
                break
            # empty source name (e.g. speaker-sink graph) — keep looking further back
            continue

    # ── nearest getGraphSettings (deviceType: 0 = microphone) line above —
    # carries Windows' own "audioEnhancementState" for the mic. The exact
    # meaning of each numeric value isn't documented here; treated as
    # 0 = Disabled, anything else = Enabled — flag this if a real capture
    # shows that guess is wrong for some value.
    audio_enhancement_state = None
    for j in range(idx - 1, max(idx - 1 - _LOOKBACK, -1), -1):
        m = RE_GET_GRAPH_SETTINGS.search(lines[j])
        if m and m.group(1) == "0":
            try:
                dev_info, _ = _json_mod.JSONDecoder().raw_decode(m.group(2))
                audio_enhancement_state = dev_info.get("audioEnhancementState")
            except Exception:
                pass
            break

    # best-effort: parse a human-readable accent name out of the model, e.g.
    # "ar-filipino-wb-c" -> "filipino"
    accent_guess = ""
    if model_name:
        am = re.match(r"ar-([a-z]+)-", model_name)
        if am:
            accent_guess = am.group(1)

    return {
        "ts_str":        ts_str,
        "ts_dt":         ts_dt,
        "model_name":    model_name,
        "accent_guess":  accent_guess,
        "mic_name":      mic_name,
        "mic_sample_rate": mic_sample_rate,
        "audio_enhancement_state": audio_enhancement_state,
        "file":          fp,
        "recording_line":     lines[idx].rstrip(),
        "proc_settings_line": ar_line,
        "graph_line":         graph_line,
    }


print("✅ Helper functions loaded.")


In [ ]:
RUN_CPU_AND_OVERWRITE = True
VOLUME_PRODUCT_THRESHOLD = 0.1    # initial; adjustable in the Timeline tab; fixed in Last Calls
VOLUME_MIN_THRESHOLD     = 0.25   # flag if device or other-device volume < this
RUN_VOLUME_MIC        = True
RUN_VOLUME_SPEAKER    = True
RUN_KR_APP_SETUP      = True

# ── Run the analysis ─────────────────────────────────────────

start_dt = parse_ts(START_TIME) if START_TIME else None
end_dt   = parse_ts(END_TIME)   if END_TIME   else None

if start_dt and end_dt and start_dt > end_dt:
    raise ValueError("START_TIME must be earlier than END_TIME.")

# ── dm_reporting check ───────────────────────────────────────
dm_is_empty, dm_report_path = check_dm_reporting(FOLDER)

# ── Photos/videos accidentally included with the report ────────
media_images_found, media_videos_found = find_top_level_media_files(FOLDER)

# ── Headsets: connected at submission vs seen-but-disconnected ──
submitted_mic_devices, submitted_spk_devices, other_mic_devices, other_spk_devices = collect_device_report(FOLDER)

# ── systeminfo.json check ────────────────────────────────────
import json as _json

FREE_MEMORY_THRESHOLD = 1_500_000_000   # 1.5 GB in bytes

device_map = parse_device_map(FOLDER)

_sysinfo_path = Path(FOLDER) / "systeminfo.json"
sysinfo            = None
memory_free        = None
memory_free_flagged = False
cpu_title           = ""

if _sysinfo_path.exists():
    try:
        with open(_sysinfo_path, "r", encoding="utf-8", errors="replace") as _fh:
            sysinfo = _json.load(_fh)
        memory_free = sysinfo.get("memory", {}).get("free")
        if memory_free is not None:
            memory_free_flagged = memory_free < FREE_MEMORY_THRESHOLD
        cpu_title = sysinfo.get("cpu", {}).get("title", "")
    except Exception as _e:
        sysinfo = None
        print(f"⚠️  Could not parse systeminfo.json: {_e}")

# Resolve file lists
cpu_files      = resolve_files(FOLDER, "*native_utils*.log") + resolve_files(FOLDER, "*native_utils*.txt")
media_sp_files = resolve_files(FOLDER, "*kr_media_sp*.log")  + resolve_files(FOLDER, "*kr_media_sp*.txt")
audio_dm_files = resolve_files(FOLDER, "*kr_audio_dm*.log")  + resolve_files(FOLDER, "*kr_audio_dm*.txt")

# ── Conditional runs ─────────────────────────────────────────
cpu_readings = []; cpu_total_blocks = cpu_in_range = 0; cpu_spikes = []
mem_readings = []; mem_total_blocks = mem_in_range = 0; mem_spikes = []
ow_flagged   = []; ow_total_blocks  = ow_in_range  = 0
vol_flagged  = []; vol_total        = vol_in_range  = 0; vol_all = []
kr_app_files = []; kr_unhealthy = []; kr_overload = []; kr_mute = []; kr_ducking = []; kr_sessions = []
kr_mic_dumps = []; kr_spk_dumps = []; kr_mic_calls = []; kr_spk_calls = []; kr_call_ends = {}
kr_mic_calls_raw = []; kr_spk_calls_raw = []
testnc_info = None
last_session_username = None
last_session_version = None
last_log_submit = None   # (ts_dt, ts_str, is_actual_submission) or None
testnc_audio_sample_rate = None

if not dm_is_empty:
    if RUN_CPU_AND_OVERWRITE:
        cpu_readings, cpu_total_blocks, cpu_in_range = collect_cpu_readings(cpu_files, start_dt, end_dt)
        cpu_spikes = find_cpu_spikes(cpu_readings, THRESHOLD, CONSECUTIVE)
        mem_readings, mem_total_blocks, mem_in_range = collect_memory_readings(cpu_files, start_dt, end_dt)
        mem_spikes = find_cpu_spikes(mem_readings, MEMORY_THRESHOLD, CONSECUTIVE)
        ow_flagged, ow_total_blocks, ow_in_range = find_overwrite_flags(media_sp_files, OVERWRITE_THRESHOLD, start_dt, end_dt)
        print(f"✅ CPU analysis complete      — {len(cpu_spikes)} spike event(s) found.")
        print(f"✅ Memory analysis complete   — {len(mem_spikes)} spike event(s) found.")
        print(f"✅ Overwrite analysis complete — {len(ow_flagged)} flagged block(s) found.")

    if RUN_VOLUME_MIC or RUN_VOLUME_SPEAKER:
        _vol_types = set()
        if RUN_VOLUME_MIC:     _vol_types.add(0)
        if RUN_VOLUME_SPEAKER: _vol_types.add(1)
        vol_flagged, vol_total, vol_in_range = collect_volume_events(
            audio_dm_files, VOLUME_PRODUCT_THRESHOLD, start_dt, end_dt, _vol_types
        )
        # also collect ALL volume events (threshold=1.0 catches everything)
        vol_all, _, _ = collect_volume_events(
            audio_dm_files, 1.0, start_dt, end_dt, _vol_types
        )
        mic_flagged = [e for e in vol_flagged if e["dev_type"] == 0]
        spk_flagged = [e for e in vol_flagged if e["dev_type"] == 1]
        print(f"✅ Volume analysis complete   — {len(mic_flagged)} mic / {len(spk_flagged)} speaker event(s) flagged.")

    if RUN_KR_APP_SETUP:
        kr_app_files = resolve_files(FOLDER, "*kr_app*.log") + resolve_files(FOLDER, "*kr_app*.txt")
        # exclude files that match kr_app_cpu or similar non-system kr_app variants
        kr_app_files = [f for f in kr_app_files if "kr_app" in f.name.lower()]
        kr_unhealthy, kr_overload, kr_mute, kr_ducking, kr_sessions = collect_kr_app_system(kr_app_files, start_dt, end_dt)
        kr_mic_dumps, kr_spk_dumps = collect_kr_app_dumpstreams(kr_app_files, start_dt, end_dt)
        kr_mic_calls, kr_spk_calls, kr_call_ends, kr_mic_calls_raw, kr_spk_calls_raw = collect_kr_app_calls(kr_app_files, start_dt, end_dt)
        testnc_info = collect_testnc_last(kr_app_files)
        last_session_username = extract_last_session_username(kr_app_files)
        last_session_version = extract_last_session_version(kr_app_files)
        last_log_submit = extract_last_log_submission_time(kr_app_files)
        # A TestNC ("Test my mic") session always ships with the recorded
        # audio captures alongside the logs, so only look for the file when
        # a TestNC session was actually found.
        if testnc_info is not None:
            testnc_audio_sample_rate = get_wav_sample_rate(Path(FOLDER) / "before_proc.wav")
        print(f"✅ Kr_app analysis complete  — {len(kr_unhealthy)} unhealthy / "
              f"{len(kr_overload)} overload / {len(kr_mute)} mute / "
              f"{len(kr_ducking)} ducking / "
              f"{len(kr_mic_dumps)+len(kr_spk_dumps)} dumpstreams / "
              f"{len(kr_mic_calls)+len(kr_spk_calls)} call starts found.")
        print("✅ TestNC analysis complete  — "
              + (f"last session at {testnc_info['ts_str']}" if testnc_info else "no TestNC session found"))
else:
    print("⚠️  dm_reporting.txt is missing or empty — skipping log analysis.")


In [ ]:
# ── Build & display tabbed report ────────────────────────────
from IPython.display import display, HTML
import html as _html

SEP  = "=" * 72
SEP2 = "-" * 72

HTML_PAYLOAD = "__HTML__"   # sentinel: payload is raw HTML, skip <pre> wrapping

# Shown on the left side of every pane's Top/Bottom/sort bar so it's visible
# no matter which tab you're on. Prefer the actual report-submission time
# (from the last request-stats block in kr_app logs); if that block doesn't
# exist, fall back to the single latest timestamp recorded anywhere in the
# kr_app logs so this never has to say "not found" in practice.
if last_log_submit:
    _lls_dt, _lls_str, _lls_is_submit = last_log_submit
    _log_submit_label = ("Log submitted: " if _lls_is_submit else "Last log activity: ") + _lls_str
else:
    _log_submit_label = "Log submitted: (not found)"

# ── Tab-name health colors ─────────────────────────────────────
# A handful of tabs get their own button label colored green/red as a
# quick at-a-glance status indicator, independent of whatever tab you're
# currently looking at. All of this is keyed off the same report-submission
# reference timestamp shown in the pane header above.
_TAB_GREEN = "#6fcf97"
_TAB_RED   = "#ef4444"

_report_ref_dt    = last_log_submit[0] if last_log_submit else None
_recent_cutoff_dt = (_report_ref_dt - timedelta(days=4)) if _report_ref_dt else None


def _any_recent(ts_list):
    """True if any timestamp in ts_list falls within the last 4 days of the
    report reference time. If we have no reference time at all (couldn't
    determine when the report was submitted), treat any event as recent
    rather than silently hiding a real issue."""
    if not ts_list:
        return False
    if _recent_cutoff_dt is None:
        return True
    return any(ts >= _recent_cutoff_dt for ts in ts_list)


def _sort_events(lst):
    """Reverse a list of events if SORT_DESCENDING is True.
    Works on any list — reversal is in-place safe via slicing.
    """
    return list(reversed(lst)) if SORT_DESCENDING else lst



def _tab_report(tabs):
    """
    tabs: list of either:
      (label, lines_list)                       — simple tab
      (label, [(sub_label, lines_list), ...])   — nested sub-tabs
    lines_list = None → disabled (greyed out).
    """
    uid = "kr_tabs"

    css = (
        "<style>"
        # ── outer widget ──────────────────────────────────────
        ".kr-widget{font-family:monospace}"
        # ── outer tab bar (sticky) ────────────────────────────
        ".kr-tab-bar{display:flex;gap:0;flex-wrap:wrap;"
        "position:sticky;top:0;z-index:999;background:#181818;"
        "border-bottom:2px solid #555;padding-top:4px}"
        ".kr-tab-btn{padding:6px 16px;cursor:pointer;"
        "border:1px solid #555;border-bottom:none;"
        "background:#2b2b2b;color:#ccc;"
        "margin-right:2px;border-radius:4px 4px 0 0;"
        "font-size:13px;user-select:none}"
        ".kr-tab-btn:hover{background:#3c3c3c}"
        ".kr-tab-btn.active{background:#1e1e1e;color:#fff;"
        "border-bottom:2px solid #1e1e1e;margin-bottom:-2px}"
        ".kr-tab-btn.disabled{opacity:0.4;cursor:default;pointer-events:none}"
        ".kr-tab-pane{display:none}"
        ".kr-tab-pane.active{display:block}"
        # ── inner (nested) sub-tab bar ────────────────────────
        ".kr-sub-bar{display:flex;gap:0;flex-wrap:wrap;"
        "border-bottom:2px solid #444;margin-bottom:0;padding-top:6px;"
        "background:#1a1a1a}"
        ".kr-sub-btn{padding:4px 14px;cursor:pointer;"
        "border:1px solid #444;border-bottom:none;"
        "background:#222;color:#bbb;"
        "margin-right:2px;border-radius:4px 4px 0 0;"
        "font-size:12px;user-select:none}"
        ".kr-sub-btn:hover{background:#2e2e2e}"
        ".kr-sub-btn.active{background:#1e1e1e;color:#fff;"
        "border-bottom:2px solid #1e1e1e;margin-bottom:-2px}"
        ".kr-sub-pane{display:none}"
        ".kr-sub-pane.active{display:block}"
        # ── pane header: scroll buttons ───────────────────────
        ".kr-pane-hdr{display:flex;justify-content:space-between;align-items:center;gap:6px;"
        "padding:4px 6px;background:#252525;"
        "border:1px solid #444;border-bottom:none;border-radius:4px 4px 0 0}"
        ".kr-pane-hdr-left{font-size:12px;color:#999;font-family:monospace;padding-left:4px;white-space:nowrap}"
        ".kr-pane-hdr-right{display:flex;gap:6px}"
        ".kr-scroll-btn{padding:2px 12px;font-size:12px;cursor:pointer;"
        "background:#2b2b2b;color:#ccc;border:1px solid #555;"
        "border-radius:3px;user-select:none}"
        ".kr-scroll-btn:hover{background:#3c3c3c}"
        # ── pre ───────────────────────────────────────────────
        ".kr-tab-pane pre,.kr-sub-pane pre"
        "{font-family:monospace;font-size:12.5px;"
        "line-height:1.5;background:#1e1e1e;color:#d4d4d4;"
        "border:1px solid #444;border-radius:0 0 4px 4px;"
        "padding:14px 18px;overflow-x:auto;overflow-y:auto;"
        "max-height:600px;margin:0;"
        "scrollbar-width:thin;scrollbar-color:#555 #1e1e1e}"
        ".kr-tab-pane pre::-webkit-scrollbar,.kr-sub-pane pre::-webkit-scrollbar"
        "{width:8px;height:8px}"
        ".kr-tab-pane pre::-webkit-scrollbar-track,"
        ".kr-sub-pane pre::-webkit-scrollbar-track{background:#1e1e1e}"
        ".kr-tab-pane pre::-webkit-scrollbar-thumb,"
        ".kr-sub-pane pre::-webkit-scrollbar-thumb"
        "{background:#555;border-radius:4px}"
        ".kr-tab-pane pre::-webkit-scrollbar-thumb:hover,"
        ".kr-sub-pane pre::-webkit-scrollbar-thumb:hover{background:#777}"
        "</style>"
    )

    def _make_pane_html(pre_id, content, is_html=False):
        """Scroll buttons + sort toggle + content."""
        top_js = "document.getElementById('" + pre_id + "').scrollTop=0"
        bot_js = ("document.getElementById('" + pre_id + "').scrollTop"
                  + "=document.getElementById('" + pre_id + "').scrollHeight")
        # Sort JS is fully inline on the button — no window. globals, works across Jupyter outputs.
        # btn.parentElement = kr-pane-hdr div; .nextElementSibling = the content wrapper.
        sort_onclick = (
            "var b=this;"
            "var nowAsc=b.getAttribute('data-dir')==='asc';"
            "b.setAttribute('data-dir',nowAsc?'desc':'asc');"
            "b.textContent=nowAsc?'\u2191 Oldest first':'\u2193 Newest first';"
            "var wrap=b.closest('.kr-pane-hdr').nextElementSibling;"
            "if(!wrap)return;"
            "var tbs=wrap.querySelectorAll('table[data-sortable] tbody');"
            "tbs.forEach(function(tb){"
            "var rs=Array.from(tb.querySelectorAll('tr[data-ts]'));"
            "rs.forEach(function(r){r._savedDisp=r.style.display;});"
            "rs.sort(function(a,b){"
            "return nowAsc?(a.dataset.ts<b.dataset.ts?1:-1):(a.dataset.ts>b.dataset.ts?1:-1);});"
            "rs.forEach(function(r){tb.appendChild(r);r.style.display=r._savedDisp;});});"
        )
        hdr = (
            '<div class="kr-pane-hdr">'
            + '<span class="kr-pane-hdr-left">' + _html.escape(_log_submit_label) + '</span>'
            + '<div class="kr-pane-hdr-right">'
            + '<button class="kr-scroll-btn" onclick="' + top_js + '">&#9650; Top</button>'
            + '<button class="kr-scroll-btn" onclick="' + bot_js + '">&#9660; Bottom</button>'
            + '<button class="kr-scroll-btn" data-dir="desc" onclick="' + sort_onclick + '">&#8593; Oldest first</button>'
            + '</div>'
            + '</div>'
        )
        if is_html:
            return (hdr
                    + '<div id="' + pre_id + '" style="overflow-y:auto;overflow-x:auto;max-height:600px;'
                    + 'background:#1e1e1e;border:1px solid #444;border-radius:0 0 4px 4px;'
                    + 'scrollbar-width:thin;scrollbar-color:#555 #1e1e1e">' + content + '</div>')
        # Plain-text tabs (Summary & Driver, Ducking/Unhealthy, TestNC) rely on
        # the .kr-tab-pane pre CSS rule for their dark background, but Jupyter's
        # own notebook styling can win that specificity fight and render them
        # light. Force the same dark theme inline so it can't be overridden.
        _pre_inline = (
            "background:#1e1e1e !important;color:#d4d4d4 !important;"
            "font-family:monospace;font-size:12.5px;line-height:1.5;"
            "border:1px solid #444;border-radius:0 0 4px 4px;"
            "padding:14px 18px;overflow-x:auto;overflow-y:auto;"
            "max-height:600px;margin:0"
        )
        return hdr + '<pre id="' + pre_id + '" style="' + _pre_inline + '">' + content + '</pre>'
    buttons = []
    panes   = []
    first_active = None

    for idx, entry in enumerate(tabs):
        if len(entry) == 3:
            label, payload, tab_color = entry
        else:
            label, payload = entry
            tab_color = None
        tid      = uid + "-" + str(idx)
        disabled = payload is None
        # detect nested: payload is a list of (sub_label, lines) tuples
        nested   = (not disabled
                    and isinstance(payload, list)
                    and len(payload) > 0
                    and isinstance(payload[0], tuple))
        active   = (not disabled and first_active is None)
        if active:
            first_active = idx

        btn_cls = ("kr-tab-btn"
                   + (" active"   if active   else "")
                   + (" disabled" if disabled else ""))
        onclick = ("" if disabled
                   else "krSwitchTab('" + uid + "'," + str(idx) + ")")
        label_html = _html.escape(label)
        if tab_color:
            # inline color always wins over the button's own color rules
            # (including .active), so the health indicator stays visible
            # whether or not that tab is currently selected.
            label_html = '<span style="color:' + tab_color + '">' + label_html + '</span>'
        buttons.append(
            '<button class="' + btn_cls + '" id="' + tid + '-btn"'
            + ' onclick="' + onclick + '">' + label_html + '</button>'
        )

        pane_cls = "kr-tab-pane" + (" active" if active else "")

        if disabled:
            inner = "<pre>(disabled)</pre>"
        elif nested:
            # build inner sub-tab bar + sub-panes
            sub_buttons = []
            sub_panes   = []
            for sidx, (slabel, slines) in enumerate(payload):
                stid      = tid + "-s" + str(sidx)
                sactive   = (sidx == 0)
                sbtn_cls  = "kr-sub-btn" + (" active" if sactive else "")
                sonclick  = "krSwitchSub('" + tid + "'," + str(sidx) + ")"
                sub_buttons.append(
                    '<button class="' + sbtn_cls + '" id="' + stid + '-btn"'
                    + ' onclick="' + sonclick + '">' + _html.escape(slabel) + '</button>'
                )
                spane_cls = "kr-sub-pane" + (" active" if sactive else "")
                s_is_html = (isinstance(slines, list)
                             and len(slines) == 2
                             and slines[0] == HTML_PAYLOAD)
                if s_is_html:
                    scontent = slines[1]
                elif slines is not None:
                    scontent = _html.escape("\n".join(slines))
                else:
                    scontent = "(disabled)"
                sub_panes.append(
                    '<div class="' + spane_cls + '" id="' + stid + '-pane">'
                    + _make_pane_html(stid + "-pre", scontent, s_is_html)
                    + '</div>'
                )
            inner = (
                '<div class="kr-sub-bar">' + "".join(sub_buttons) + '</div>'
                + "".join(sub_panes)
            )
        else:
            # check for HTML_PAYLOAD sentinel on outer tabs too
            p_is_html = (isinstance(payload, list)
                         and len(payload) == 2
                         and payload[0] == HTML_PAYLOAD)
            if p_is_html:
                content = payload[1]
            else:
                content = _html.escape("\n".join(payload))
            inner = _make_pane_html(tid + "-pre", content, p_is_html)

        panes.append(
            '<div class="' + pane_cls + '" id="' + tid + '-pane">'
            + inner + '</div>'
        )

    js = (
        "<script>"
        "function krSwitchTab(uid,idx){"
        "document.querySelectorAll('[id^=\"'+uid+'-\"][id$=\"-btn\"]')"
        ".forEach(b=>{if(!b.id.includes('-s'))b.classList.remove('active')});"
        "document.querySelectorAll('[id^=\"'+uid+'-\"][id$=\"-pane\"]')"
        ".forEach(p=>{if(!p.id.includes('-s'))p.classList.remove('active')});"
        "document.getElementById(uid+'-'+idx+'-btn').classList.add('active');"
        "document.getElementById(uid+'-'+idx+'-pane').classList.add('active');}"
        "function krSwitchSub(pid,idx){"
        "document.querySelectorAll('[id^=\"'+pid+'-s\"][id$=\"-btn\"]').forEach(b=>b.classList.remove('active'));"
        "document.querySelectorAll('[id^=\"'+pid+'-s\"][id$=\"-pane\"]').forEach(p=>p.classList.remove('active'));"
        "document.getElementById(pid+'-s'+idx+'-btn').classList.add('active');"
        "document.getElementById(pid+'-s'+idx+'-pane').classList.add('active');}"
        "</script>"
    )

    html_out = (
        '<div class="kr-widget">'
        + css
        + '<div class="kr-tab-bar">' + "".join(buttons) + '</div>'
        + "".join(panes)
        + js
        + '</div>'
    )
    display(HTML(html_out))


# ═════════════════════════════════════════════════════════════
# Build each tab's content as a lines list
# ═════════════════════════════════════════════════════════════

def _fmt_bytes(b):
    """Format bytes as a human-readable GB/MB string."""
    if b >= 1_000_000_000:
        return f"{b / 1_073_741_824:.2f} GB  ({b:,} bytes)"
    return f"{b / 1_048_576:.1f} MB  ({b:,} bytes)"


def _render_lines_colored(line_color_pairs):
    """Render (text, color_or_None) pairs as monospace lines inside a dark
    box matching the other tabs — raw HTML (not the escaped <pre> path) so
    individual lines can be colored, e.g. a red warning for a stale version."""
    wrap_style = ("font-family:monospace;font-size:12.5px;line-height:1.5;"
                  "color:#d4d4d4;padding:14px 18px;white-space:pre-wrap")
    parts = []
    for text, color in line_color_pairs:
        style = "display:block"
        if color:
            style += ";color:" + color
        parts.append("<span style=\"" + style + "\">" + _html.escape(text) + "</span>")
    return "<div style=\"" + wrap_style + "\">" + "".join(parts) + "</div>"


# ── Tab 0: Summary & Driver Check ────────────────────────────
_version_mismatch = bool(last_session_version) and last_session_version != LATEST_KRISP_VERSION
_version_color = "#ff5555" if _version_mismatch else None
_version_line = f"  Krisp version (last session)   : {last_session_version or '(not found)'}"
if _version_mismatch:
    _version_line += f"   ⚠️  latest available is {LATEST_KRISP_VERSION}"

tab0_lines = [
    (SEP, None),
    ("  SUMMARY & DRIVER CHECK", None),
    (SEP, None),
    ("", None),
    ("PARAMETERS", None),
    ("-" * 40, None),
    (f"  Time range (start)             : {START_TIME if START_TIME else 'beginning of file'}", None),
    (f"  Time range (end)               : {END_TIME   if END_TIME   else 'end of file'}", None),
    (f"  Username (last session, kr_app): {last_session_username or '(not found)'}", None),
    (_version_line, _version_color),
    (f"  Latest version available       : {LATEST_KRISP_VERSION}", None),
    ("", None),
    (SEP, None),
    ("  DRIVER / DEVICE CHECK  (dm_reporting.txt)", None),
    (SEP, None),
    ("", None),
]
if dm_report_path is None or dm_is_empty:
    tab0_lines += [
        ("  ⚠️  dm_reporting.txt is missing or completely empty.", None),
        ("  This indicates a driver issue — Krisp was unable to detect audio devices.", None),
        ("", None),
    ]
else:
    tab0_lines += [
        (f"  File   : {dm_report_path}", None),
        ("  ✅ dm_reporting.txt present and contains data. Proceeding with analysis.", None),
        ("", None),
    ]

# ── Photos/videos accidentally included with the report ────────
if media_images_found or media_videos_found:
    tab0_lines += [
        (SEP, None),
        ("  PHOTOS / VIDEOS ATTACHED", None),
        (SEP, None),
        ("", None),
        ("  ⚠️  The customer included photo/video files alongside the logs:", "#fbbf24"),
        ("", None),
    ]
    for _name in media_images_found:
        tab0_lines.append((f"    🖼  {_name}", None))
    for _name in media_videos_found:
        tab0_lines.append((f"    🎬 {_name}", None))
    tab0_lines.append(("", None))

tab0 = [HTML_PAYLOAD, _render_lines_colored(tab0_lines)]

if dm_report_path is None or dm_is_empty:
    _tab_report([("Summary & Driver", tab0)])
    raise SystemExit("Stopping: dm_reporting.txt is missing or empty.")

# ── Tab: Headsets ──────────────────────────────────────────────
# Two views per device type: what was actually connected at the moment the
# report was submitted (dm_reporting.txt's live snapshot), and what else
# shows up earlier in the logs but wasn't connected anymore by submission
# time — useful for spotting "customer switched headsets mid-session" or
# "the headset they're complaining about wasn't even plugged in when they
# sent the report" style cases.
def _headsets_render(submitted, other, dev_label):
    """submitted/other are sorted lists of device NAMES (already deduplicated
    upstream in collect_device_report — the same physical device can
    re-enumerate under a new device ID each reconnect, so grouping by name
    instead of ID avoids showing the same headset multiple times)."""
    lines = [
        (SEP, None),
        (f"  HEADSETS — {dev_label.upper()}", None),
        (SEP, None),
        ("", None),
        ("CONNECTED AT REPORT SUBMISSION", None),
        ("-" * 40, None),
        ("", None),
    ]
    if submitted:
        for name in submitted:
            lines.append((f"  • {name}", None))
    else:
        lines.append((f"  (none — no {dev_label.lower()} was connected when the report was submitted)", _TAB_RED))
    lines.append(("", None))
    lines += [
        (SEP2, None),
        ("SEEN IN LOGS, BUT NOT CONNECTED AT SUBMISSION", None),
        (SEP2, None),
        ("", None),
    ]
    if other:
        for name in other:
            lines.append((f"  • {name}", None))
    else:
        lines.append((f"  (none — no other {dev_label.lower()}s were seen earlier in the session)", None))
    lines.append(("", None))
    return _render_lines_colored(lines)

tab_headsets_mic = [HTML_PAYLOAD, _headsets_render(submitted_mic_devices, other_mic_devices, "Microphone")]
tab_headsets_spk = [HTML_PAYLOAD, _headsets_render(submitted_spk_devices, other_spk_devices, "Speaker")]

# Red if either mic or speaker had nothing connected at submission time
# (dm_reporting.txt's live snapshot) — that's the case worth checking.
# Green only when both sides had something connected.
_headsets_color = _TAB_RED if (not submitted_mic_devices or not submitted_spk_devices) else _TAB_GREEN

# ── Tab 1: CPU & Memory ───────────────────────────────────────
if RUN_CPU_AND_OVERWRITE:
    # cpu title banner
    if cpu_title:
        cpu_title_html = ("<div style=\"padding:6px 10px;background:#1a1a1a;"
                          "font-family:monospace;font-size:13px;color:#d4d4d4\">"
                          "&#128187; " + _html.escape(cpu_title) + "</div>")
    else:
        cpu_title_html = ""

    # memory section (plain text — not threshold-dependent)
    if sysinfo is None:
        mem_html = "<pre style=\"background:#1e1e1e;color:#fb923c;padding:10px\">⚠ systeminfo.json not found.</pre>"
    elif memory_free is None:
        mem_html = "<pre style=\"background:#1e1e1e;color:#fb923c;padding:10px\">⚠ memory.free not found in systeminfo.json.</pre>"
    elif memory_free_flagged:
        mem_html = ('<pre style="background:#2a0a0a;color:#f28b82;padding:10px">⚠ FREE MEMORY LOW: '
                    + _fmt_bytes(memory_free) + " (threshold: " + _fmt_bytes(FREE_MEMORY_THRESHOLD) + ")</pre>")
    else:
        mem_html = ('<pre style="background:#1e1e1e;color:#6fcf97;padding:10px">✅ Free memory: '
                    + _fmt_bytes(memory_free) + "</pre>")

    # CPU spikes — embed all data as data-* so JS can re-filter on threshold/consecutive change
    th_s = "position:sticky;top:0;z-index:2;padding:4px 10px;text-align:left;background:#252525;color:#fff;border-bottom:2px solid #555"
    td_s = "padding:3px 10px;white-space:nowrap;font-family:monospace;font-size:12.5px;text-align:left"
    cpu_tbl = ("<table id=\"cpu-tbl\" data-sortable=\"1\" style=\"width:100%;border-collapse:collapse;font-family:monospace;font-size:12.5px\"><thead><tr>"
               + "".join("<th style=\""+th_s+"\">"+h+"</th>" for h in ("Start Timestamp","End Timestamp","Readings","Peak %","Avg %","CPU Row"))
               + "</tr></thead><tbody>")
    for run in cpu_readings:  # cpu_readings = list of individual (ts_str,ts_dt,idx,val)
        pass  # placeholder — actual spike rows below
    cpu_tbl_rows = ""
    for run in _sort_events(cpu_spikes):
        first_ts = run[0][0]; last_ts = run[-1][0]
        values = [v for _, _, _, v in run]
        peak = max(values); avg = sum(values)/len(values)
        cpu_row = " | ".join(f"{v:.1f}" for v in values)
        # data-peak and data-count for JS filtering
        _cpu_a = "data-peak=" + chr(34) + f"{peak:.2f}" + chr(34) + " data-count=" + chr(34) + str(len(run)) + chr(34) + " data-ts=" + chr(34) + first_ts + chr(34)
        _cpu_s = "style=" + chr(34) + "background:#1e1e1e;border-bottom:1px solid #2a2a2a" + chr(34)
        cpu_tbl_rows += "<tr " + _cpu_a + " " + _cpu_s + ">"
        cpu_tbl_rows += "".join("<td style=" + chr(34) + td_s + ";color:#d4d4d4" + chr(34) + ">" + _html.escape(str(v)) + "</td>" for v in (first_ts, last_ts, str(len(run)), f"{peak:.2f}%", f"{avg:.2f}%", cpu_row))
        cpu_tbl_rows += "</tr>"
    cpu_tbl += cpu_tbl_rows + "</tbody></table>"
    cpu_note = ('<div id="cpu-note" style="font-family:monospace;font-size:12px;color:#aaa;padding:6px 10px;background:#1a1a1a">'
                f"Total blocks: {cpu_total_blocks} | In range: {cpu_in_range} | "
                f"Total samples: {len(cpu_readings)} | Spike events: {len(cpu_spikes)} "
                '</div>')
    cpu_js = ("<script>"
              "function krCpuFilter(){"
              "var thr=parseFloat(document.getElementById('kr-cpu-thr').value);"
              "var con=parseInt(document.getElementById('kr-consec').value);"
              "if(isNaN(thr))thr=0;if(isNaN(con))con=0;"
              "var rows=document.querySelectorAll('#cpu-tbl tbody tr');"
              "rows.forEach(function(r){"
              "var ok=parseFloat(r.dataset.peak)>thr&&parseInt(r.dataset.count)>=con;"
              "r.style.display=ok?''  :'none';});"
              "}"
              "</script>")
    tab1 = [HTML_PAYLOAD, cpu_title_html + mem_html + cpu_note + cpu_tbl + cpu_js]
else:
    tab1 = None

# ── Tab 2: Overwrite ──────────────────────────────────────────
if RUN_CPU_AND_OVERWRITE:
    th_s2 = "position:sticky;top:0;z-index:2;padding:4px 10px;text-align:left;background:#252525;color:#fff;border-bottom:2px solid #555"
    td_s2 = "padding:3px 10px;white-space:nowrap;font-family:monospace;font-size:12.5px;text-align:left;color:#d4d4d4"
    ow_tbl = ("<table id=\"ow-tbl\" data-sortable=\"1\" style=\"width:100%;border-collapse:collapse;font-family:monospace;font-size:12.5px\"><thead><tr>"
              + "".join("<th style=\""+th_s2+"\">"+h+"</th>" for h in ("Timestamp","Write Cnt","Overwrite Cnt","Ratio %"))
              + "</tr></thead><tbody>")
    for ts_str, _, w_cnt, ow_cnt, ratio, w_line, ow_line, fp in _sort_events(ow_flagged):
        _ow_a = "data-wcnt=" + chr(34) + str(w_cnt) + chr(34) + " data-owcnt=" + chr(34) + str(ow_cnt) + chr(34) + " data-ts=" + chr(34) + ts_str + chr(34)
        _ow_s = "style=" + chr(34) + "background:#1e1e1e;border-bottom:1px solid #2a2a2a" + chr(34)
        ow_tbl += "<tr " + _ow_a + " " + _ow_s + ">"
        ow_tbl += "".join("<td style=" + chr(34) + td_s2 + chr(34) + ">" + _html.escape(str(v)) + "</td>" for v in (ts_str, str(w_cnt), str(ow_cnt), f"{ratio:.1f}"))
        ow_tbl += "</tr>"
    ow_tbl += "</tbody></table>"
    ow_note = ('<div id="ow-note" style="font-family:monospace;font-size:12px;color:#aaa;padding:6px 10px;background:#1a1a1a">'
               f"Total Capturer::Stop blocks: {ow_total_blocks} | In range: {ow_in_range} | "
               f"All overwrite events: {len(ow_flagged)} "
               '</div>')
    ow_js = ("<script>"
             "function krOwFilter(){"
             "var thr=parseFloat(document.getElementById('kr-ow-thr').value);"
             "if(isNaN(thr))thr=0;"
             "var rows=document.querySelectorAll('#ow-tbl tbody tr');"
             "rows.forEach(function(r){"
             "var w=parseInt(r.dataset.wcnt)||0;var ow=parseInt(r.dataset.owcnt)||0;"
             "var ok=w>0&&(ow/w*100)>thr;r.style.display=ok?''  :'none';});"
             "}"
             "</script>")
    tab2 = [HTML_PAYLOAD, ow_note + ow_tbl + ow_js]
else:
    tab2 = None

# ── shared helper (used by Kr_app — Timeline below) ──────────
import re as _re_vol

def _extract_dev_id(line, key):
    m = _re_vol.search(key + r":\s*(\{[^,]+\})", line)
    return m.group(1).strip() if m else ""


# ── Tab 5: Kr_app — Ducking, Unhealthy ───────────────────────
if RUN_KR_APP_SETUP:

    tab5_sys = [
        SEP,
        "  KR_APP — DUCKING / UNHEALTHY",
        SEP,
        "",
    ]

    # ── 1. Ducking ───────────────────────────────────────────
    tab5_sys += [SEP2, "  1. DUCKING MODE", SEP2, ""]
    if kr_ducking:
        last = kr_ducking[-1]
        ts_str, _, error_line, duck_line = last
        tab5_sys.append(f"  Latest ducking mode entry  :  {ts_str}")
        tab5_sys.append("")
        if error_line:
            tab5_sys.append("  ⚠️  Registry error detected immediately before ducking line:")
            tab5_sys.append("")
            tab5_sys.append(f"  {error_line}")
        tab5_sys.append(f"  {duck_line}")
        tab5_sys.append("")
    else:
        tab5_sys += ["  No ducking mode entries found.", ""]

    # ── 2. Unhealthy ─────────────────────────────────────────
    tab5_sys += [SEP2, "  2. UNHEALTHY STATE", SEP2, ""]
    if kr_unhealthy:
        tab5_sys.append(f"  ⚠️  {len(kr_unhealthy)} unhealthy event(s) detected.")
        tab5_sys.append("")
        for i, (ts_str, raw, dev_name) in enumerate(_sort_events(kr_unhealthy), 1):
            tab5_sys += [
                f"  Event #{i}",
                f"    Timestamp : {ts_str}",
                *([ f"    Device    : {dev_name}" ] if dev_name else []),
                f"    {raw}",
                "",
            ]
    else:
        tab5_sys += ["  ✅ No unhealthy state events found.", ""]

else:
    tab5_sys = None

# ── HTML Settings Panel ──────────────────────────────────────
# ── Render all tabs ───────────────────────────────────────────

# ── Tab: Kr_app — Timeline (Calls + Streams combined) ────────
if RUN_KR_APP_SETUP:
    # App names treated as OS/background noise rather than real calls —
    # shared by the "Hide System Settings & rundll32" row-toggle (is_ss,
    # below) AND by _last_n_real_calls' call-skipping logic further down.
    # These two used to be separate hardcoded lists that drifted out of sync
    # (_OS_NOISE_APPS was missing "systemsettings"/"system", so a literal
    # "SystemSettings" AppName — the actual format Windows uses, no space —
    # wasn't excluded when picking the last N real calls, even though the
    # row-hide toggle correctly matched it everywhere else). Single source
    # of truth now so that can't happen again.
    _HIDDEN_APP_NAMES = {"systemsettings", "system settings", "rundll32", "system", "livecaptions"}

    # colors for timeline rows
    _CLR_CALL    = "#1a3a2a"   # dark green tint  → Call Start
    _CLR_SESSION = "#1a1a3a"   # dark indigo      → Session Start
    _CLR_TYPE_SESSION  = "#818cf8"  # bright indigo label
    _CLR_CALL_END      = "#0a2a1a"   # dark teal   → Call End
    _CLR_TYPE_CALL_END = "#34d399"   # bright teal
    _CLR_DUMP = "#1a2a3a"   # dark blue tint   → DumpStream
    _CLR_HDR  = "#252525"   # header row
    _CLR_TXT  = "#d4d4d4"
    _CLR_TYPE_CALL = "#6fcf97"   # bright green label
    _CLR_TYPE_DUMP = "#56aef7"   # bright blue label

    def _td(val, color=None, bold=False):
        style = "padding:3px 10px;white-space:nowrap;font-family:monospace;font-size:12.5px;text-align:left;vertical-align:top"
        if color:
            style += ";color:" + color
        if bold:
            style += ";font-weight:bold"
        return "<td style=\"" + style + "\">" + _html.escape(str(val)) + "</td>"

    _CLR_MUTE = "#2a1a3a"        # dark purple → Mute
    _CLR_VOL  = "#2a2a10"        # dark yellow → Volume
    _CLR_TYPE_MUTE = "#c084fc"   # bright purple
    _CLR_TYPE_VOL  = "#fbbf24"   # bright yellow

    _CLR_CPU = "#3a1010"          # dark red    → CPU spike
    _CLR_MEM = "#3a1a10"          # dark orange → Memory spike
    _CLR_TYPE_CPU = "#f87171"     # bright red
    _CLR_TYPE_MEM = "#fb923c"     # bright orange
    _CLR_TYPE_AC  = "#c084fc"     # bright purple → Accent Conversion sub-line
    _CLR_OW      = "#3a2a10"      # dark amber  → Overwrite
    _CLR_TYPE_OW  = "#facc15"     # bright amber
    _CLR_OVL      = "#3a1a2a"     # dark magenta → System Overload
    _CLR_TYPE_OVL = "#e879f9"     # bright magenta
    _CLR_SELF     = "#0f5c38"     # bright green → the call this mini-timeline is about
    _CLR_SELF_END = "#0a4530"

    # "red = issue" — used only in Last Calls (issue_mode=True), never in the
    # whole-log Timeline, per the user's request that red stays exclusive to
    # Last Calls.
    _CLR_ISSUE      = "#3a0f0f"   # dark red
    _CLR_TYPE_ISSUE = "#ef4444"   # bright red

    # "green = ok" — the counterpart to _CLR_ISSUE, also Last Calls-only.
    # Every non-issue row (call boundaries, healthy dumpstreams, healthy
    # volume readings, etc.) renders in this single green instead of its
    # normal Timeline-tab color, so the Last Calls tab reads as a simple
    # green/red health scan.
    _CLR_OK      = "#123a1a"   # dark green
    _CLR_TYPE_OK = "#4ade80"   # bright green

    _CALL_KINDS = ("call", "call_end", "self_call", "self_call_end")

    # CHANGE 2: keyword to detect krisp devices in timeline
    def _is_krisp_dev(name): return "krisp" in name.lower()

    def _fmt_dur(secs_str):
        """Convert seconds string to 'Xm Ys' or 'Ys'."""
        try:
            s = int(secs_str)
            if s >= 60:
                return f"{s//60}m {s%60}s"
            return f"{s}s"
        except (ValueError, TypeError):
            return secs_str + "s" if secs_str else ""

    def _fmt_khz(sr):
        """Format a raw sample-rate value like 48000 as '48k' (or '44.1k'
        for non-round-thousands rates). Returns '' for missing/invalid input."""
        try:
            v = float(sr) / 1000.0
        except (ValueError, TypeError):
            return ""
        return f"{v:.0f}k" if v == int(v) else f"{v:.1f}k"

    # ═══════════════════════════════════════════════════════════
    # Shared row builders — each returns a list of
    #   (ts_dt, ts_str, kind_label, app, extra/dev, kind_key, flag, call_id)
    # so the same rows can feed either the whole-log Timeline tabs or a
    # single call's time-windowed mini-timeline (Last Calls tab).
    # ═══════════════════════════════════════════════════════════
    def _build_call_rows(calls, call_num_map, dev_tag, call_ends):
        rows = []
        for ts_str, ts_dt, app_name, call_id, nc, mute, dev_name, bvc, ts_full, ar_language, sample_rate in calls:
            cnum = call_num_map.get(call_id, "")
            # sample_rate appended last (index 9) so existing _cp[0..8] indexing
            # elsewhere stays valid regardless of whether it's present.
            call_extra = "||".join(["call", str(cnum), dev_name, nc, mute, bvc, ts_full, ar_language, dev_tag, str(sample_rate)])
            rows.append((ts_dt, ts_str, f"Call Start #{cnum}", app_name, call_extra, "call", True, call_id))
            if call_id in call_ends:
                end_ts_str, end_ts_dt, duration, final_nc, final_mute, end_ts_full = call_ends[call_id]
                # only add end if it comes after (use microseconds for same-second calls)
                if end_ts_full >= ts_full:
                    _elapsed = int((end_ts_dt - ts_dt).total_seconds())
                    _dur_label = _fmt_dur(str(_elapsed))
                    end_extra = "||".join(["call_end", str(cnum), "", final_nc, final_mute, "", end_ts_full, "", dev_tag, ""])
                    rows.append((end_ts_dt, end_ts_str, f"Call End #{cnum}   {_dur_label}", app_name, end_extra, "call_end", True, call_id))
        return rows

    def _build_dump_rows(dumps):
        rows = []
        for ts_str, ts_dt, entries in dumps:
            for device_name, app_name in entries:
                # CHANGE 2: flag non-krisp dump streams
                is_ok = _is_krisp_dev(device_name)
                rows.append((ts_dt, ts_str, "DumpStream", app_name, device_name, "dump", is_ok, ""))
        return rows

    def _build_mute_rows(mutes):
        """mutes: list of (ts_str, ts_dt, dvt, dev_name, from_m, to_m, raw) — kr_mute format."""
        rows = []
        for ts_str, ts_dt, dvt, dev_name, from_m, to_m, raw in mutes:
            direction = "Muted" if to_m else "Unmuted"
            rows.append((ts_dt, ts_str, "Mute — " + direction, "", dev_name, "mute", to_m, ""))
        return rows

    def _build_vol_rows(vol_evs):
        rows = []
        for ev in vol_evs:
            _did  = _extract_dev_id(ev["raw_line"], "DeviceId")
            _odid = _extract_dev_id(ev["raw_line"], "OtherDeviceId")
            _dn   = device_map.get(_did,  _did)  if _did  else ""
            _odn  = device_map.get(_odid, _odid) if _odid else ""
            is_ok = (ev["product"] >= VOLUME_PRODUCT_THRESHOLD
                     and ev["volume"] >= VOLUME_MIN_THRESHOLD
                     and ev["other_vol"] >= VOLUME_MIN_THRESHOLD)
            rows.append((ev["ts_dt"], ev["ts_str"], "Volume", "",
                         "||".join([_dn, str(ev["volume"]), _odn, str(ev["other_vol"]), f"{ev['product']:.4f}"]),
                         "vol", is_ok, ""))
        return rows

    def _build_session_rows(sessions):
        rows = []
        for ts_str, ts_dt, instance, raw in sessions:
            label_txt = "Session Start" + (" — " + instance if instance else "")
            rows.append((ts_dt, ts_str, label_txt, "", "", "session", True, ""))
        return rows

    def _build_cpu_rows():
        rows = []
        for run in cpu_spikes:
            _first_ts_str, _first_ts_dt = run[0][0], run[0][1]
            _last_ts_str = run[-1][0]
            _values = [v for _, _, _, v in run]
            _peak = max(_values); _avg = sum(_values) / len(_values)
            _detail = f"Peak {_peak:.1f}% | Avg {_avg:.1f}% | {len(run)} reading(s) | {_first_ts_str} → {_last_ts_str}"
            rows.append((_first_ts_dt, _first_ts_str, "CPU Spike", "", _detail, "cpu", True, ""))
        return rows

    def _build_mem_rows():
        rows = []
        for run in mem_spikes:
            _first_ts_str, _first_ts_dt = run[0][0], run[0][1]
            _last_ts_str = run[-1][0]
            _values = [v for _, _, _, v in run]
            _peak = max(_values); _avg = sum(_values) / len(_values)
            _detail = f"Peak {_peak:.1f}% | Avg {_avg:.1f}% | {len(run)} reading(s) | {_first_ts_str} → {_last_ts_str}"
            rows.append((_first_ts_dt, _first_ts_str, "Memory Spike", "", _detail, "mem", True, ""))
        return rows

    def _build_ow_rows():
        rows = []
        for _ow_ts_str, _ow_ts_dt, _w_cnt, _ow_cnt, _ratio, _w_line, _ow_line, _fp in ow_flagged:
            _detail = f"Write: {_w_cnt} | Overwrite: {_ow_cnt} | Ratio: {_ratio:.1f}%"
            rows.append((_ow_ts_dt, _ow_ts_str, "Overwrite", "", _detail, "overwrite", True, ""))
        return rows

    def _build_overload_rows():
        """Hard/Overloaded system-overload events — previously their own section
        in the Kr_app — Ducking/Unhealthy tab; now surfaced in the Timeline and
        Last Calls tabs instead."""
        rows = []
        for _ol_ts_str, _ol_ts_dt, _filter_id, _state, _raw in kr_overload:
            if _filter_id.lower() != "hard" or _state.lower() != "overloaded":
                continue
            _detail = f"FilterId: {_filter_id} | State: {_state}"
            rows.append((_ol_ts_dt, _ol_ts_str, "System Overload", "", _detail, "overload", True, ""))
        return rows

    # ═══════════════════════════════════════════════════════════
    # Shared renderer: sort + toggle buttons + table HTML + JS.
    # show_dev_tag=True annotates OTHER calls (not the "self" call) with
    # their device type + short call id — used by the Last Calls mini-timeline.
    # ═══════════════════════════════════════════════════════════
    def _render_event_table(rows, uid, label, show_dev_tag=False, all_visible=False, issue_mode=False, show_overwrite=True):
        # min-width:100% + width:max-content (instead of a hard width:100%) lets
        # the table grow past the container's width when columns need more room
        # (e.g. long device names + the AC-language sub-line) instead of being
        # squeezed down and having nowrap text spill over into neighboring
        # cells. The wrapping div below adds overflow-x:auto so the extra
        # width becomes a horizontal scrollbar rather than an overlap.
        tbl_style = ("min-width:100%;width:max-content;border-collapse:collapse;"
                     "font-family:monospace;font-size:12.5px;color:" + _CLR_TXT)
        th_style  = ("position:sticky;top:40px;z-index:2;padding:4px 10px;text-align:left;"
                     "background:" + _CLR_HDR + ";color:#fff;"
                     "border-bottom:2px solid #555")

        if not rows:
            html = ("<p style=\"font-family:monospace;color:" + _CLR_TXT
                    + ";padding:12px\">No events found for "
                    + _html.escape(label) + ".</p>")
            return [HTML_PAYLOAD, html]

        # Use ts_full (microsecond) as tiebreaker so sub-second call pairs sort correctly
        def _row_ts_key(r):
            ts_dt, ts_str, _, _, extra, kind_key, _flag, _cid = r
            if kind_key in _CALL_KINDS and "||" in extra:
                parts = extra.split("||")
                if len(parts) > 6 and parts[6]:
                    return parts[6]   # ts_full string e.g. "05/08/26 04:47:03.705758"
            return ts_str             # fallback: second-precision string
        rows = sorted(rows, key=lambda r: (r[0], _row_ts_key(r)), reverse=SORT_DESCENDING)

        # toggle buttons — kind_key may be a comma-separated list (e.g. "call,call_end")
        # active=True  → visible by default, button click HIDES the rows (Calls, DumpStreams)
        # active=False → hidden by default,  button click SHOWS the rows (Mute, Volume, CPU, Memory, Overwrite)
        def _toggle_btn(tid, kind_key, label_txt, color, active=False):
            op = "1" if active else "0.5"
            da = "1" if active else "0"
            return (
                '<button id="' + tid + '" ' +
                'onclick="krTlToggle(\'' + uid + '\',\'' + kind_key + '\',\'' + tid + '\')" ' +
                'style="margin:4px 4px 4px 0;padding:3px 12px;font-size:12px;cursor:pointer;' +
                'background:#2b2b2b;color:' + color + ';border:1px solid ' + color + ';border-radius:3px' +
                ';opacity:' + op + '" data-active="' + da + '">' + label_txt + '</button>'
            )

        tl_thr_id = uid + "-vt"
        tl_mvthr_id = uid + "-mvt"
        tl_inp_s  = "padding:2px 5px;width:55px;background:#2b2b2b;color:#d4d4d4;border:1px solid #555;border-radius:3px;font-family:monospace;font-size:12px"
        tl_lbl_s  = "font-family:monospace;font-size:12px;color:#aaa;margin-right:4px"

        # Interactive Product/Min-vol threshold inputs + live re-color, now
        # available in both the Timeline tabs and Last Calls — just with
        # colors matching whichever scheme that tab uses (amber/yellow for
        # Timeline, red/green for Last Calls' issue_mode).
        bad_vol_id = uid + "-badvol"
        vol_btn_id = uid + "-tvol"
        _low_bg, _low_tc = (_CLR_ISSUE, _CLR_TYPE_ISSUE) if issue_mode else ("#2a1a00", "#fb923c")
        _ok_bg,  _ok_tc  = (_CLR_OK, _CLR_TYPE_OK)       if issue_mode else ("#1e1e0a", "#fde68a")
        tl_recolor = (
            "var vt=parseFloat(document.getElementById('" + tl_thr_id   + "').value)||0;"
            "var mv=parseFloat(document.getElementById('" + tl_mvthr_id + "').value)||0;"
            "document.querySelectorAll('#" + uid + "-tbl tbody tr').forEach(function(r){"
            "if(r.getAttribute('data-kind')!=='vol')return;"
            "var tds=r.querySelectorAll('td');"
            "var vol=parseFloat(tds[4]?tds[4].textContent:0)||0;"
            "var ovol=parseFloat(tds[5]?tds[5].textContent:0)||0;"
            "var prod=parseFloat(tds[6]?tds[6].textContent:0)||0;"
            "var low=prod<vt||vol<mv||ovol<mv;"
            "r.style.background=low?'" + _low_bg + "':'" + _ok_bg + "';"
            "r.querySelectorAll('td').forEach(function(td){td.style.color=low?'" + _low_tc + "':'" + _ok_tc + "';});});"
            # keep the "bad volume only" filter in sync if it's engaged
            "if(document.getElementById('" + bad_vol_id + "').dataset.active==='1'){"
            "krVolBadOnly('" + uid + "','" + tl_thr_id + "','" + tl_mvthr_id + "','"
            + vol_btn_id + "','" + bad_vol_id + "',true);}"
        )
        threshold_controls = (
            "<span style=\"" + tl_lbl_s + ";margin-left:8px\">Product &lt;:</span>"
            + "<input id=\"" + tl_thr_id + "\" type=\"number\" step=\"any\" "
            + "value=\"" + str(VOLUME_PRODUCT_THRESHOLD) + "\" "
            + "oninput=\"" + tl_recolor + "\" "
            + "style=\"" + tl_inp_s + "\">"
            + "<span style=\"" + tl_lbl_s + "\">Min vol &lt;:</span>"
            + "<input id=\"" + tl_mvthr_id + "\" type=\"number\" step=\"any\" "
            + "value=\"" + str(VOLUME_MIN_THRESHOLD) + "\" "
            + "oninput=\"" + tl_recolor + "\" "
            + "style=\"" + tl_inp_s + "\">"
            # shows only below-threshold volume rows (hides the passing ones),
            # re-evaluated live against whatever the two inputs above are set to
            + "<button id=\"" + bad_vol_id + "\" "
            + "onclick=\"krVolBadOnly('" + uid + "','" + tl_thr_id + "','" + tl_mvthr_id + "','"
            + vol_btn_id + "','" + bad_vol_id + "')\" "
            + "style=\"margin:4px 4px 4px 0;padding:3px 12px;font-size:12px;cursor:pointer;"
            + "background:#2b2b2b;color:#fb923c;border:1px solid #fb923c;border-radius:3px;opacity:0.5\" "
            + "data-active=\"0\">Bad volume only</button>"
        )

        # Last Calls (issue_mode) uses one flat toggle color instead of each
        # button matching its row's category hue — rows are already just
        # green/red there, so per-category toggle colors would be a mismatched
        # legend. The Timeline tabs keep their original per-category colors.
        _CLR_TOGGLE = "#60a5fa"   # uniform blue, Last Calls toggles only
        def _tg(default_color):
            return _CLR_TOGGLE if issue_mode else default_color

        toggles = (
            "<div style=\"position:sticky;top:0;z-index:998;padding:6px 8px;"
            "background:#181818;border-bottom:1px solid #444;display:flex;align-items:center;gap:10px;flex-wrap:wrap\">"
            + "<span style=\"" + tl_lbl_s + "\">Toggle:</span>"
            + _toggle_btn(uid+"-tcall", "call,call_end", "Calls",         _tg(_CLR_TYPE_CALL), active=True)
            # DumpStreams default to visible in Timeline (issue_mode=False) but
            # hidden-by-default in Last Calls (issue_mode=True) — a red dump
            # there just means "not routed through Krisp", not "something is
            # wrong", so it shouldn't be the first thing you see in a triage view.
            + _toggle_btn(uid+"-tdump", "dump",          "DumpStreams",   _tg(_CLR_TYPE_DUMP), active=(not issue_mode))
            + _toggle_btn(uid+"-tmute", "mute",           "Mute events",   _tg(_CLR_TYPE_MUTE), active=all_visible)
            + _toggle_btn(uid+"-tvol",  "vol",            "Volume events", _tg(_CLR_TYPE_VOL),  active=all_visible)
            + _toggle_btn(uid+"-tcpu",  "cpu",            "CPU spikes",    _tg(_CLR_TYPE_CPU),  active=all_visible)
            + _toggle_btn(uid+"-tmem",  "mem",            "Memory spikes", _tg(_CLR_TYPE_MEM),  active=all_visible)
            + _toggle_btn(uid+"-tovl",  "overload",       "Overloads",     _tg(_CLR_TYPE_OVL),  active=all_visible)
            + (_toggle_btn(uid+"-tow", "overwrite",      "Overwrites",    _tg(_CLR_TYPE_OW),   active=all_visible) if show_overwrite else "")
            + threshold_controls
            + "<button id=\"" + uid + "-ssbtn\" "
            + "onclick=\"krTlSsToggle('" + uid + "')\" "
            + "style=\"margin-left:8px;padding:3px 12px;font-size:12px;cursor:pointer;background:#2b2b2b;color:#ccc;border:1px solid #555;border-radius:3px\" "
            + "data-active=\"1\">Hide System Settings &amp; rundll32</button>"
            + "</div>"
        )

        html  = "<table id=\"" + uid + "-tbl\" data-sortable=\"1\" style=\"" + tbl_style + "\"><thead><tr>"
        html += "".join("<th style=" + chr(34) + th_style + chr(34) + ">" + h + "</th>" for h in ("Timestamp","Type","App","Device","Volume/NC","Other/Muted","Product/BVC"))
        html += "</tr></thead><tbody>"

        # Which row kinds start hidden must mirror each toggle button's own
        # `active=` above exactly, or the button state and the actual rows
        # go out of sync (e.g. a button reading "click to show" while the
        # rows it controls are already visible). "call"/"call_end" are never
        # hidden. "dump" follows the Timeline-vs-Last-Calls split. Everything
        # else just follows all_visible like before.
        _kind_starts_hidden = {
            "dump": issue_mode,
            "mute": not all_visible,
            "vol": not all_visible,
            "cpu": not all_visible,
            "mem": not all_visible,
            "overwrite": not all_visible,
            "overload": not all_visible,
        }

        for _, ts_str, kind, app, dev, kind_key, flag, call_id in rows:
            hide_css = (";display:none" if _kind_starts_hidden.get(kind_key, False) else "")
            self_border = ""
            if kind_key == "call_end":
                bg, tc = (_CLR_OK, _CLR_TYPE_OK) if issue_mode else (_CLR_CALL_END, _CLR_TYPE_CALL_END)
            elif kind_key == "call":
                bg, tc = (_CLR_OK, _CLR_TYPE_OK) if issue_mode else (_CLR_CALL, _CLR_TYPE_CALL)
            elif kind_key == "self_call":
                bg, tc = _CLR_SELF, "#ffffff"
                self_border = ";border-left:4px solid #fff"
            elif kind_key == "self_call_end":
                bg, tc = _CLR_SELF_END, "#ffffff"
                self_border = ";border-left:4px solid #fff"
            elif kind_key == "dump":
                # CHANGE 2: non-krisp dump = darker blue (or red, in issue_mode)
                if flag:
                    bg, tc = (_CLR_OK, _CLR_TYPE_OK) if issue_mode else (_CLR_DUMP, _CLR_TYPE_DUMP)
                elif issue_mode:
                    bg, tc = _CLR_ISSUE, _CLR_TYPE_ISSUE
                else:
                    bg, tc = "#0a1a2a", "#93c5fd"
            elif kind_key == "session":
                bg, tc = (_CLR_OK, _CLR_TYPE_OK) if issue_mode else (_CLR_SESSION, _CLR_TYPE_SESSION)
            elif kind_key == "mute":
                # In Last Calls this row only ever appears when muted (an
                # issue) — always red. In the Timeline, keep the original
                # muted=pink-purple / unmuted=light-purple scheme.
                if issue_mode:
                    bg, tc = _CLR_ISSUE, _CLR_TYPE_ISSUE
                else:
                    bg = "#2a1020" if flag else "#1a1a30"
                    tc = "#f472b6" if flag else "#a78bfa"
            elif kind_key == "cpu":
                bg, tc = (_CLR_ISSUE, _CLR_TYPE_ISSUE) if issue_mode else (_CLR_CPU, _CLR_TYPE_CPU)
            elif kind_key == "mem":
                bg, tc = (_CLR_ISSUE, _CLR_TYPE_ISSUE) if issue_mode else (_CLR_MEM, _CLR_TYPE_MEM)
            elif kind_key == "overwrite":
                bg, tc = (_CLR_ISSUE, _CLR_TYPE_ISSUE) if issue_mode else (_CLR_OW, _CLR_TYPE_OW)
            elif kind_key == "overload":
                bg, tc = (_CLR_ISSUE, _CLR_TYPE_ISSUE) if issue_mode else (_CLR_OVL, _CLR_TYPE_OVL)
            else:  # vol
                # below threshold = red in Last Calls, amber in Timeline; above threshold = pale yellow (both)
                if not flag:
                    bg, tc = (_CLR_ISSUE, _CLR_TYPE_ISSUE) if issue_mode else ("#2a1a00", "#fb923c")
                else:
                    bg, tc = (_CLR_OK, _CLR_TYPE_OK) if issue_mode else ("#1e1e0a", "#fde68a")
            is_ss = (app.lower() in _HIDDEN_APP_NAMES)
            ss_attr = " data-ss=\"1\"" if is_ss else ""
            # System Settings / rundll32 rows are visible by default; the
            # "Hide System Settings & rundll32" button hides them on click.
            row_style = "background:" + bg + ";border-bottom:1px solid #2a2a2a" + hide_css + self_border
            _dts = ts_str
            if kind_key in _CALL_KINDS and "||" in dev:
                _dts_parts = dev.split("||")
                if len(_dts_parts) > 6 and _dts_parts[6]:
                    _dts = _dts_parts[6]
            html += "<tr style=\"" + row_style + "\" data-kind=\"" + kind_key + "\" data-ts=\"" + _dts + "\"" + ss_attr + ">"
            html += _td(ts_str)

            # Type column — self rows are colored only now (no star; the
            # green/white-border highlighting already marks "this is the
            # call in question"); other calls optionally get a (device type,
            # short call id) tag; any call with accent conversion active gets
            # " · <language> AC" appended right here instead of a separate
            # line under the device name.
            _ar_lang = ""
            if kind_key in _CALL_KINDS and "||" in dev:
                _ar_probe = dev.split("||")
                _ar_lang = _ar_probe[7] if len(_ar_probe) > 7 else ""

            if kind_key in ("self_call", "self_call_end"):
                _kind_disp = kind
            elif kind_key in ("call", "call_end") and show_dev_tag:
                _tag_full = ""
                if "||" in dev:
                    _parts_probe = dev.split("||")
                    _traw = _parts_probe[8] if len(_parts_probe) > 8 else ""
                    _tag_full = {"mic": "microphone", "spk": "speaker"}.get(_traw, _traw)
                _short_id = call_id[:8] if call_id else ""
                _suffix = f" ({_tag_full}, {_short_id})" if _tag_full else ""
                _kind_disp = kind + _suffix
            else:
                _kind_disp = kind
            if kind_key in ("call", "self_call") and _ar_lang:
                _kind_disp += f" · {_ar_lang} AC"
            html += _td(_kind_disp, color=tc, bold=True)
            html += _td(app)
            if kind_key in _CALL_KINDS and "||" in dev:
                _cp = dev.split("||")  # [type, cnum, dev_name, nc, mute, bvc, ts_full, ar_language, dev_tag, sample_rate]
                _sr_disp = _fmt_khz(_cp[9]) if len(_cp) > 9 else ""
                _dev_cell = _html.escape(_cp[2])
                if _sr_disp:
                    # sample rate goes right next to the mic/speaker name, same line
                    _dev_cell += (" <span style=\"color:#888;font-size:11px\">"
                                   + _html.escape(_sr_disp) + "</span>")
                _dev_td_style = ("padding:3px 10px;white-space:nowrap;font-family:monospace;"
                                  "font-size:12.5px;text-align:left;vertical-align:top;color:" + tc)
                html += "<td style=\"" + _dev_td_style + "\">" + _dev_cell + "</td>"
                html += _td(_cp[3], color=tc)  # Volume/NC: NC state
                html += _td(_cp[4], color=tc)  # Other/Muted: Mute state
                html += _td(_cp[5], color=tc)  # Product/BVC: BVC state
            elif kind_key == "vol" and "||" in dev:
                _vp = dev.split("||")
                _dn, _vol, _odn, _ovol = _vp[0], _vp[1], _vp[2], _vp[3]
                if "krisp" in _dn.lower() and "krisp" not in _odn.lower():
                    _dn, _vol, _odn, _ovol = _odn, _ovol, _dn, _vol
                html += _td(_dn)    # Device/NC col
                html += _td(_vol)   # Volume/Muted col
                html += _td(_ovol)  # OtherVolume col
                html += _td(_vp[4]) # Product/BVC col
            else:
                html += _td(dev)
                html += _td("")
                html += _td("")
                html += _td("")
            html += "</tr>"

        html += "</tbody></table>"

        js = (
            '<script>'
            'function krTlToggle(uid,kind,btnId){'
            'var btn=document.getElementById(btnId);'
            'var turningOn=(btn.dataset.active!=="1");'
            'btn.dataset.active=turningOn?"1":"0";'
            'btn.style.opacity=turningOn?"1":"0.5";'
            'var kinds=kind.split(",");'
            'var rows=document.getElementById(uid+"-tbl").querySelectorAll("tbody tr");'
            'rows.forEach(function(r){'
            'if(kinds.indexOf(r.getAttribute("data-kind"))!==-1){'
            'r.style.display=turningOn?"table-row":"none";}});'
            # turning off "Volume events" should also turn off "Bad volume
            # only" — it doesn't make sense left active while volume rows
            # are hidden entirely
            'if(kind==="vol"&&!turningOn){'
            'var badBtn=document.getElementById(uid+"-badvol");'
            'if(badBtn&&badBtn.dataset.active==="1"){'
            'badBtn.dataset.active="0";badBtn.style.opacity="0.5";}}'
            '}'
            'function krTlSsToggle(uid){'
            'var btn=document.getElementById(uid+"-ssbtn");'
            'var on=(btn.dataset.active==="1");'
            'document.querySelectorAll("#"+uid+"-tbl tr[data-ss]").forEach(function(r){r.style.display=on?"none":"";});'
            'btn.dataset.active=on?"0":"1";'
            'btn.textContent=on?"Show System Settings & rundll32":"Hide System Settings & rundll32";'
            '}'
            # Shows only below-threshold ("bad") volume rows, hides the
            # passing ones. Re-usable so the threshold inputs can re-apply it
            # live (forceOn=true) while it's engaged, and the button's own
            # click can toggle it on/off.
            'function krVolBadOnly(uid,thrId,mvThrId,volBtnId,btnId,forceOn){'
            'var btn=document.getElementById(btnId);'
            'var turningOn=(typeof forceOn==="boolean")?forceOn:(btn.dataset.active!=="1");'
            'btn.dataset.active=turningOn?"1":"0";'
            'btn.style.opacity=turningOn?"1":"0.5";'
            'var volBtn=document.getElementById(volBtnId);'
            'if(turningOn&&volBtn&&volBtn.dataset.active!=="1"){volBtn.click();}'
            'var vt=parseFloat(document.getElementById(thrId).value)||0;'
            'var mv=parseFloat(document.getElementById(mvThrId).value)||0;'
            'document.querySelectorAll("#"+uid+"-tbl tbody tr").forEach(function(r){'
            'if(r.getAttribute("data-kind")!=="vol")return;'
            'var tds=r.querySelectorAll("td");'
            'var vol=parseFloat(tds[4]?tds[4].textContent:0)||0;'
            'var ovol=parseFloat(tds[5]?tds[5].textContent:0)||0;'
            'var prod=parseFloat(tds[6]?tds[6].textContent:0)||0;'
            'var low=prod<vt||vol<mv||ovol<mv;'
            'if(turningOn){r.style.display=low?"table-row":"none";}'
            'else if(volBtn){r.style.display=(volBtn.dataset.active==="1")?"table-row":"none";}'
            '});'
            '}'
            '</script>'
        )

        return [HTML_PAYLOAD, toggles + html + js]

    def _timeline_tab(calls, dumps, mutes, vol_evs, sessions, call_ends, dev_type, label, call_num_map, dev_tag):
        rows = []
        rows += _build_call_rows(calls, call_num_map, dev_tag, call_ends)
        rows += _build_dump_rows(dumps)
        rows += _build_mute_rows(mutes)
        rows += _build_vol_rows(vol_evs)
        rows += _build_session_rows(sessions)
        rows += _build_cpu_rows()
        rows += _build_mem_rows()
        rows += _build_overload_rows()
        # Overwrite is shown under Microphone only.
        if dev_type == 0:
            rows += _build_ow_rows()
        uid = "tl-" + str(dev_type)
        return _render_event_table(rows, uid, label, show_overwrite=(dev_type == 0))

    # sequential call numbers (oldest start = #1), computed once and reused
    # by both the whole-log Timeline tabs and the Last Calls tab below.
    _sorted_mic_calls_for_num = sorted(kr_mic_calls, key=lambda c: c[8] if len(c) > 8 else str(c[1]))
    call_num_map_mic = {c[3]: idx + 1 for idx, c in enumerate(_sorted_mic_calls_for_num)}
    _sorted_spk_calls_for_num = sorted(kr_spk_calls, key=lambda c: c[8] if len(c) > 8 else str(c[1]))
    call_num_map_spk = {c[3]: idx + 1 for idx, c in enumerate(_sorted_spk_calls_for_num)}

    mic_mute_all = [e for e in kr_mute if e[2] == "microphone"]
    spk_mute_all = [e for e in kr_mute if e[2] == "speaker"]
    mic_vol_rows  = _sort_events([e for e in vol_all if e["dev_type"] == 0])
    spk_vol_rows  = _sort_events([e for e in vol_all if e["dev_type"] == 1])

    tab_timeline_mic = _timeline_tab(kr_mic_calls, kr_mic_dumps, mic_mute_all, mic_vol_rows, kr_sessions, kr_call_ends, 0, "microphone", call_num_map_mic, "mic")
    tab_timeline_spk = _timeline_tab(kr_spk_calls, kr_spk_dumps, spk_mute_all, spk_vol_rows, kr_sessions, kr_call_ends, 1, "speaker", call_num_map_spk, "spk")
else:
    tab_timeline_mic = None
    tab_timeline_spk = None

# ── Tab: Last Calls — detailed, time-windowed mini-timeline per call ──
if RUN_KR_APP_SETUP:
    # Pre-build the row sets once; each call's mini-timeline is just a
    # time-window slice of these. Calls, dumpstreams, mute and volume events
    # are kept separate per device type — a Microphone call's mini-timeline
    # should only ever show microphone-related events, and a Speaker call's
    # only speaker-related events. CPU/Memory/Overwrite/Session events are
    # system-wide (not tied to a device), so those stay shared.
    _all_call_rows_mic = _build_call_rows(kr_mic_calls, call_num_map_mic, "mic", kr_call_ends)
    _all_call_rows_spk = _build_call_rows(kr_spk_calls, call_num_map_spk, "spk", kr_call_ends)
    _all_dump_rows_mic = _build_dump_rows(kr_mic_dumps)
    _all_dump_rows_spk = _build_dump_rows(kr_spk_dumps)
    _all_mute_rows_mic = _build_mute_rows(mic_mute_all)
    _all_mute_rows_spk = _build_mute_rows(spk_mute_all)
    _all_vol_rows_mic  = _build_vol_rows(mic_vol_rows)
    _all_vol_rows_spk  = _build_vol_rows(spk_vol_rows)
    _all_session_rows = _build_session_rows(kr_sessions)
    _all_cpu_rows  = _build_cpu_rows()
    _all_mem_rows  = _build_mem_rows()
    _all_overload_rows = _build_overload_rows()
    _all_ow_rows   = _build_ow_rows()   # microphone tab only — see _call_detail_tab

    _log_dt_pool = (
        [c[1] for c in kr_mic_calls] + [c[1] for c in kr_spk_calls]
        + [v[1] for v in kr_call_ends.values()]
        + [d[1] for d in kr_mic_dumps] + [d[1] for d in kr_spk_dumps]
        + [m[1] for m in kr_mute]
        + [e["ts_dt"] for e in vol_all]
        + [run[-1][1] for run in cpu_spikes]
        + [run[-1][1] for run in mem_spikes]
        + [e[1] for e in ow_flagged]
    )
    LOG_MAX_DT = max(_log_dt_pool) if _log_dt_pool else datetime.max

    def _call_window_bounds(call_id, call_dt, dev_type):
        """(window_start, window_end) for a call: its own CALL_END if found,
        else the next same-device call's start, else the end of the logs."""
        if call_id in kr_call_ends:
            return call_dt, kr_call_ends[call_id][1]
        same_dev_calls = kr_mic_calls if dev_type == 0 else kr_spk_calls
        later_starts = [c[1] for c in same_dev_calls if c[1] > call_dt]
        if later_starts:
            return call_dt, min(later_starts)
        return call_dt, LOG_MAX_DT

    def _mark_self(rows, current_call_id):
        out = []
        for ts_dt, ts_str, kind, app, dev, kind_key, flag, cid in rows:
            if cid == current_call_id and kind_key == "call":
                kind_key = "self_call"
            elif cid == current_call_id and kind_key == "call_end":
                kind_key = "self_call_end"
            out.append((ts_dt, ts_str, kind, app, dev, kind_key, flag, cid))
        return out

    def _last_mute_before(dev_type_str, before_dt):
        """Most recent mute/unmute event for a device type, strictly before before_dt."""
        candidates = [e for e in kr_mute if e[2] == dev_type_str and e[1] < before_dt]
        if not candidates:
            return None
        return max(candidates, key=lambda e: e[1])

    def _last_vol_before(dev_type_int, dev_name_filter, before_dt):
        """Most recent volume-change event for a device type, strictly before before_dt.
        Prefers events matching dev_name_filter (the mic/speaker used in the call);
        falls back to any event of that device type if no name match is found."""
        candidates = [e for e in vol_all if e["dev_type"] == dev_type_int and e["ts_dt"] < before_dt]
        if not candidates:
            return None
        if dev_name_filter:
            named = []
            for e in candidates:
                _did  = _extract_dev_id(e["raw_line"], "DeviceId")
                _odid = _extract_dev_id(e["raw_line"], "OtherDeviceId")
                _dn   = (device_map.get(_did, _did) if _did else "") or ""
                _odn  = (device_map.get(_odid, _odid) if _odid else "") or ""
                if dev_name_filter.lower() in _dn.lower() or dev_name_filter.lower() in _odn.lower():
                    named.append(e)
            if named:
                candidates = named
        return max(candidates, key=lambda e: e["ts_dt"])

    def _relabel_row(row, suffix):
        ts_dt, ts_str, kind, app, dev, kind_key, flag, cid = row
        return (ts_dt, ts_str, kind + suffix, app, dev, kind_key, flag, cid)

    # Kinds that always render red in issue_mode (see the color-selection
    # block in _render_event_table), plus "dump"/"vol" which are red only
    # when their flag says so. Used to decide whether a call's "Call #N" tab
    # button itself should be flagged red.
    _ALWAYS_ISSUE_KINDS = {"mute", "cpu", "mem", "overwrite", "overload"}

    def _rows_have_issue(rows):
        for r in rows:
            kind_key, flag = r[5], r[6]
            if kind_key in _ALWAYS_ISSUE_KINDS:
                return True
            if kind_key in ("dump", "vol") and not flag:
                return True
        return False

    def _call_detail_tab(call_record, cnum, dev_type, dev_tag):
        ts_str, ts_dt, app_name, call_id, nc, mute, dev_name, bvc, ts_full, ar_language, sample_rate = call_record
        win_start, win_end = _call_window_bounds(call_id, ts_dt, dev_type)

        def _in_window(r):
            return win_start <= r[0] <= win_end

        # same-device-type only: mic call -> mic calls/dumps/mute/volume, spk -> spk
        _call_rows_src = _all_call_rows_mic if dev_type == 0 else _all_call_rows_spk
        _dump_rows_src = _all_dump_rows_mic if dev_type == 0 else _all_dump_rows_spk
        _mute_rows_src = _all_mute_rows_mic if dev_type == 0 else _all_mute_rows_spk
        _vol_rows_src  = _all_vol_rows_mic  if dev_type == 0 else _all_vol_rows_spk

        rows = []
        rows += list(filter(_in_window, _call_rows_src))
        rows += list(filter(_in_window, _dump_rows_src))
        rows += list(filter(_in_window, _mute_rows_src))
        rows += list(filter(_in_window, _vol_rows_src))
        rows += list(filter(_in_window, _all_session_rows))
        rows += list(filter(_in_window, _all_cpu_rows))
        rows += list(filter(_in_window, _all_mem_rows))
        rows += list(filter(_in_window, _all_overload_rows))
        # Overwrite is shown under Microphone only.
        if dev_type == 0:
            rows += list(filter(_in_window, _all_ow_rows))
        rows = _mark_self(rows, call_id)

        # Last mute/volume state before the call started — shown as extra
        # table rows (same Mute/Volume styling & toggles), not as prose.
        # Mute is only worth flagging when the device was left MUTED right
        # before the call — an unmuted "before call" state isn't an issue,
        # so it's simply not shown.
        _dev_type_str = "microphone" if dev_type == 0 else "speaker"
        _last_mute = _last_mute_before(_dev_type_str, ts_dt)
        _last_vol  = _last_vol_before(dev_type, dev_name, ts_dt)
        if _last_mute and _last_mute[5]:   # to_muted is True
            rows.append(_relabel_row(_build_mute_rows([_last_mute])[0], "  (last before call)"))
        if _last_vol:
            rows.append(_relabel_row(_build_vol_rows([_last_vol])[0], "  (last before call)"))

        has_issue = _rows_have_issue(rows)

        uid = "cd-" + dev_tag + "-" + str(cnum)
        return has_issue, _render_event_table(
            rows, uid, f"{dev_tag} call #{cnum}",
            show_dev_tag=False, all_visible=True, issue_mode=True,
            show_overwrite=(dev_type == 0),
        )

    _OS_NOISE_APPS = _HIDDEN_APP_NAMES

    def _last_n_real_calls(calls_raw):
        """Walk backward from the most recent call, skipping System
        Settings/rundll32 ones, until LAST_N_CALLS real calls are found (or
        the raw list runs out). This is what "last N calls" should mean when
        some of the raw last-N window are OS noise rather than real calls —
        e.g. if the last 2 of the raw last 7 are System Settings, we skip
        those 2 and reach back to the 7th to still show 5 real ones.
        Returns (real_calls_oldest_to_newest, excluded_count)."""
        _sorted = sorted(calls_raw, key=lambda c: c[8] if len(c) > 8 else str(c[1]))
        real = []
        excluded_ct = 0
        if LAST_N_CALLS > 0:
            for c in reversed(_sorted):   # newest -> oldest
                if c[2].lower() in _OS_NOISE_APPS:
                    excluded_ct += 1
                    continue
                real.append(c)
                if len(real) >= LAST_N_CALLS:
                    break
            real.reverse()                # back to oldest -> newest
        return real, excluded_ct

    def _last_calls_device_tab(calls_raw, call_num_map, dev_type, dev_tag, label):
        """Build the Microphone/Speaker sub-tab: its own inner tab bar with
        one tab per call, each showing that call's table-only mini-timeline.
        calls_raw includes System Settings/rundll32 calls (only "krisp" is
        dropped upstream) so _last_n_real_calls can walk back far enough to
        still surface LAST_N_CALLS real calls, with a banner reporting how
        many OS-noise calls were skipped to get there."""
        real, excluded_ct = _last_n_real_calls(calls_raw)
        ordered = list(reversed(real)) if SORT_DESCENDING else real

        _warning_html = ""
        if excluded_ct > 0:
            _plural = "s" if excluded_ct != 1 else ""
            _warning_html = (
                "<div style=\"padding:8px 10px;background:#3a2a10;color:#facc15;"
                "border-bottom:1px solid #444;font-family:monospace;font-size:12px\">"
                "⚠️ " + str(excluded_ct) + " OS/background call" + _plural
                + " (System Settings, LiveCaptions, rundll32, etc.) excluded from these results.</div>"
            )

        if not ordered:
            return [HTML_PAYLOAD, _warning_html
                    + "<p style=\"font-family:monospace;color:" + _CLR_TXT
                    + ";padding:12px\">No " + _html.escape(label.lower()) + " calls found.</p>"]

        pid = "lc-" + dev_tag
        # z-index above the per-call toggle bar (998) and table header (2) so
        # the call-tab bar always wins if it ever overlaps them; krCallBarFix()
        # below keeps them from overlapping in the first place by pushing the
        # toggle bar/table header down by the call-bar's actual rendered height.
        css = (
            "<style>"
            ".kr-call-bar{display:flex;gap:0;flex-wrap:wrap;position:sticky;top:0;z-index:999;"
            "border-bottom:2px solid #444;padding-top:6px;background:#1a1a1a}"
            ".kr-call-btn{padding:4px 14px;cursor:pointer;border:1px solid #444;border-bottom:none;"
            "background:#222;color:#bbb;margin-right:2px;border-radius:4px 4px 0 0;"
            "font-size:12px;user-select:none}"
            ".kr-call-btn:hover{background:#2e2e2e}"
            ".kr-call-btn.active{background:#1e1e1e;color:#fff;"
            "border-bottom:2px solid #1e1e1e;margin-bottom:-2px}"
            # issue = at least one red-flagged row inside that call's mini-timeline
            ".kr-call-btn.issue{color:" + _CLR_TYPE_ISSUE + ";border-color:" + _CLR_TYPE_ISSUE + "}"
            ".kr-call-btn.issue.active{background:" + _CLR_ISSUE + ";color:#fff;"
            "border-bottom:2px solid " + _CLR_ISSUE + "}"
            ".kr-call-pane{display:none}"
            ".kr-call-pane.active{display:block}"
            "</style>"
        )
        buttons = []
        panes = []
        for idx, c in enumerate(ordered):
            call_id = c[3]
            cnum = call_num_map.get(call_id, "")
            has_issue, (_, call_table_html) = _call_detail_tab(c, cnum, dev_type, dev_tag)
            btn_cls = ("kr-call-btn"
                       + (" active" if idx == 0 else "")
                       + (" issue" if has_issue else ""))
            buttons.append(
                '<button class="' + btn_cls + '" id="' + pid + '-' + str(idx) + '-btn" '
                + 'onclick="krSwitchCall(\'' + pid + '\',' + str(idx) + ')">'
                + "Call #" + str(cnum) + '</button>'
            )
            pane_cls = "kr-call-pane" + (" active" if idx == 0 else "")
            panes.append(
                '<div class="' + pane_cls + '" id="' + pid + '-' + str(idx) + '-pane">'
                + call_table_html + '</div>'
            )

        js = (
            "<script>"
            "function krSwitchCall(pid,idx){"
            "document.querySelectorAll('[id^=\"'+pid+'-\"][id$=\"-btn\"]').forEach(function(b){b.classList.remove('active')});"
            "document.querySelectorAll('[id^=\"'+pid+'-\"][id$=\"-pane\"]').forEach(function(p){p.classList.remove('active')});"
            "document.getElementById(pid+'-'+idx+'-btn').classList.add('active');"
            "document.getElementById(pid+'-'+idx+'-pane').classList.add('active');"
            "}"
            "</script>"
        )

        # Each call pane's own toggle bar (position:sticky;top:0;z-index:998,
        # built by _render_event_table) and table header (top:40px) share the
        # same scrolling ancestor as .kr-call-bar (the outer pane's overflow
        # box), so all three would stick at overlapping offsets. krCallBarFix
        # measures the call-bar's real rendered height and pushes the toggle
        # bar/table header down so the call-tab bar stays visible and
        # clickable above them no matter how far you scroll.
        fix_js = (
            "<script>"
            "function krCallBarFix(pid){"
            "var bar=document.getElementById(pid+'-bar');if(!bar)return;"
            "var H=bar.getBoundingClientRect().height;"
            "document.querySelectorAll('[id^=\"'+pid+'-\"][id$=\"-pane\"]').forEach(function(pane){"
            "var tgl=pane.querySelector('div[style*=\"z-index:998\"]');"
            "if(tgl){tgl.style.top=H+'px';"
            "var tH=tgl.getBoundingClientRect().height;"
            "pane.querySelectorAll('th[style*=\"top:40px\"]').forEach(function(th){th.style.top=(H+tH)+'px';});"
            "}"
            "});"
            "}"
            "krCallBarFix('" + pid + "');"
            "window.addEventListener('resize',function(){krCallBarFix('" + pid + "');});"
            "</script>"
        )

        html = (css + _warning_html + '<div class="kr-call-bar" id="' + pid + '-bar">' + "".join(buttons) + '</div>'
                + "".join(panes) + js + fix_js)
        return [HTML_PAYLOAD, html]

    tab_last_calls_mic = _last_calls_device_tab(kr_mic_calls_raw, call_num_map_mic, 0, "mic", "Microphone")
    tab_last_calls_spk = _last_calls_device_tab(kr_spk_calls_raw, call_num_map_spk, 1, "spk", "Speaker")
else:
    tab_last_calls_mic = None
    tab_last_calls_spk = None

# ── Tab: TestNC — last "Test my mic" recording session ───────
if RUN_KR_APP_SETUP:
    _LOW_SR_THRESHOLD = 16000   # below this -> warn in red

    def _testnc_render(line_color_pairs):
        """Render (text, color_or_None) pairs as monospace lines inside a
        dark box matching the other tabs — built as raw HTML (not the plain
        escaped <pre> path) specifically so individual lines can be colored,
        e.g. a red warning line for a too-low sample rate."""
        wrap_style = ("font-family:monospace;font-size:12.5px;line-height:1.5;"
                      "color:#d4d4d4;padding:14px 18px;white-space:pre-wrap")
        parts = []
        for text, color in line_color_pairs:
            style = "display:block"
            if color:
                style += ";color:" + color
            parts.append("<span style=\"" + style + "\">" + _html.escape(text) + "</span>")
        return "<div style=\"" + wrap_style + "\">" + "".join(parts) + "</div>"

    # A TestNC session only counts as "current" if it happened on the same
    # calendar day the report was submitted — an older session (e.g. the
    # customer tested their mic two days before submitting) is treated the
    # same as having no TestNC data at all: hide it and flag the tab red.
    _testnc_stale = (
        testnc_info is not None
        and _report_ref_dt is not None
        and testnc_info["ts_dt"].date() != _report_ref_dt.date()
    )

    if testnc_info is None:
        _testnc_color = _TAB_RED
        tab_testnc = [HTML_PAYLOAD, _testnc_render([
            (SEP, None),
            ("  TESTNC — LAST RECORDING SESSION", None),
            (SEP, None),
            ("", None),
            ("  ⚠️ No TestNC (\"Test my mic\") session found in these logs.", _TAB_RED),
            ("", None),
        ])]
    elif _testnc_stale:
        _testnc_color = _TAB_RED
        _tn_date_str     = testnc_info["ts_dt"].strftime("%m/%d/%y")
        _submit_date_str = _report_ref_dt.strftime("%m/%d/%y")
        tab_testnc = [HTML_PAYLOAD, _testnc_render([
            (SEP, None),
            ("  TESTNC — LAST RECORDING SESSION", None),
            (SEP, None),
            ("", None),
            ("  ⚠️ No TestNC session on the day the report was submitted.", _TAB_RED),
            ("", None),
            (f"  Last TestNC session was on {_tn_date_str}, but the report was submitted on {_submit_date_str}.", None),
            ("  Not showing that session's data here since it's outside the reporting day.", None),
            ("", None),
        ])]
    else:
        _testnc_color = _TAB_GREEN
        _tn = testnc_info
        _tn_dev_sr = _fmt_khz(_tn.get('mic_sample_rate', ''))
        # Audio-file sample rate, read straight from before_proc.wav's header.
        # A TestNC session always has the recorded captures sitting next to
        # the logs, so this is only attempted when testnc_info was found
        # (see testnc_audio_sample_rate in the analysis cell above).
        _tn_audio_sr_disp = _fmt_khz(testnc_audio_sample_rate) if testnc_audio_sample_rate else ""
        _low_sr = bool(testnc_audio_sample_rate) and testnc_audio_sample_rate < _LOW_SR_THRESHOLD
        _sr_line_color = _CLR_TYPE_ISSUE if _low_sr else None
        _sr_line_text = f"  Sample rate: {_tn_audio_sr_disp}" + ("   ⚠️ below 16k" if _low_sr else "")

        # Windows' own mic enhancement toggle, from the getGraphSettings line
        # a few lines before onStartRecording. NOTE: the exact meaning of
        # each numeric audioEnhancementState value isn't documented here —
        # 0 is treated as Disabled and anything else as Enabled; flag this
        # if a real capture shows that guess is wrong.
        _aes = _tn.get('audio_enhancement_state')
        _aes_line = None
        if _aes is not None:
            _aes_label = "Enabled" if _aes != 0 else "Disabled"
            _aes_line = f"  Audio enhancements: {_aes_label} (raw value: {_aes})"

        _lines = [
            (SEP, None),
            ("  TESTNC — LAST RECORDING SESSION", None),
            (SEP, None),
            ("", None),
            (f"  Recording started at: {_tn['ts_str']}", None),
        ]
        if _aes_line:
            _lines.append((_aes_line, None))
        if _tn_audio_sr_disp:
            _lines.append((_sr_line_text, _sr_line_color))
        _lines.append(("", None))
        _lines += [
            (SEP2, None),
            ("  Details", None),
            (SEP2, None),
            ("", None),
            ("  • " + (_tn['mic_name'] or '(not found)'), None),
            *([("  • " + _tn_dev_sr, None)] if _tn_dev_sr else []),
            ("  • " + (_tn['model_name'] or '(none / AC not active)'), None),
            ("", None),
        ]
        tab_testnc = [HTML_PAYLOAD, _testnc_render(_lines)]
else:
    tab_testnc = None
    _testnc_color = None

# ── CPU & Memory / Overwrite / Unhealthy tab-name colors ──────
# Green = nothing recent to worry about (including: no cases at all, or only
# cases older than 4 days before the report was submitted). Red = at least
# one qualifying case within the last 4 days. A tab only gets a color at all
# if it's actually enabled (has content) — disabled tabs stay neutral/grey.
_cpu_mem_color   = None
_overwrite_color = None
_unhealthy_color = None

if tab1 is not None:
    _recent_cpu = _any_recent([run[-1][1] for run in cpu_spikes])
    # Memory *spike* events never affect this color — only the standalone
    # "free memory low" check (systeminfo.json, same one behind the red
    # "FREE MEMORY LOW" banner already shown inside this tab) does.
    _cpu_mem_color = _TAB_RED if (_recent_cpu or memory_free_flagged) else _TAB_GREEN

if tab2 is not None:
    _recent_ow       = _any_recent([e[1] for e in ow_flagged])
    _overwrite_color = _TAB_RED if _recent_ow else _TAB_GREEN

if tab5_sys is not None:
    # Ducking-mode entries never affect this color — only unhealthy-state events do.
    _unhealthy_color = _TAB_RED if kr_unhealthy else _TAB_GREEN

_tab_report([
    ("Summary & Driver",        tab0),
    ("Headsets", [
        ("Microphone",  tab_headsets_mic),
        ("Speaker",     tab_headsets_spk),
    ], _headsets_color),
    ("CPU & Memory",            tab1, _cpu_mem_color),
    ("Overwrite",               tab2, _overwrite_color),
    ("Kr_app — Ducking/Unhealthy", tab5_sys, _unhealthy_color),
    ("TestNC",                  tab_testnc, _testnc_color),
    ("Kr_app — Timeline", [
        ("Microphone",  tab_timeline_mic),
        ("Speaker",     tab_timeline_spk),
    ]),
    ("Last Calls", [
        ("Microphone",  tab_last_calls_mic),
        ("Speaker",     tab_last_calls_spk),
    ]),
])

In [ ]:
# for scrolling